In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

print("Proyecto:", PROJECT_ROOT)
print("Datos:", DATA_RAW)

Proyecto: /Users/felipesulez/Documents/motor-asignacion-comercial
Datos: /Users/felipesulez/Documents/motor-asignacion-comercial/data/raw


In [4]:
print("Contenido de data/raw:")

for file in DATA_RAW.iterdir():
    print("-", file.name)

Contenido de data/raw:
- .gitkeep
- data-opcion-a (1).zip


In [7]:
import zipfile

zip_path = DATA_RAW / "data-opcion-a (1).zip"
EXTRACTED_DATA = DATA_RAW / "data-opcion-a"

EXTRACTED_DATA.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(EXTRACTED_DATA)

print("Datos extraidos en:", EXTRACTED_DATA)

Datos extraidos en: /Users/felipesulez/Documents/motor-asignacion-comercial/data/raw/data-opcion-a


In [8]:
for file in EXTRACTED_DATA.rglob("*"):
    if file.is_file():
        print("-", file.relative_to(EXTRACTED_DATA))

- data-opcion-a/equipos.csv
- data-opcion-a/registros.csv
- data-opcion-a/actividad.csv
- data-opcion-a/ausencias.csv
- data-opcion-a/usuarios.csv


1. Determinar las rutas de los archivos descomprimidos csv


In [12]:
DATA_DIR = EXTRACTED_DATA / "data-opcion-a"

print("Directorio de datos", DATA_DIR)

Directorio de datos /Users/felipesulez/Documents/motor-asignacion-comercial/data/raw/data-opcion-a/data-opcion-a


In [13]:
print("Archivos disponibles")

for file in DATA_DIR.iterdir():
    print("-", file.name)

Archivos disponibles
- equipos.csv
- registros.csv
- actividad.csv
- ausencias.csv
- usuarios.csv


## 1. Carga de los csv


In [16]:
actividad = pd.read_csv(DATA_DIR / "actividad.csv")
ausencias = pd.read_csv(DATA_DIR / "ausencias.csv")
equipos = pd.read_csv(DATA_DIR / "equipos.csv")
registros = pd.read_csv(DATA_DIR / "registros.csv")
usuarios = pd.read_csv(DATA_DIR / "usuarios.csv")

print("Datasets cargados correctamente")

Datasets cargados correctamente


In [17]:
datasets = {
    "actividad": actividad,
    "ausencias": ausencias,
    "equipos": equipos,
    "registros": registros,
    "usuarios": usuarios,
}

## 2. Exploraion inicial de los datos

Antes de definir las reglas del motor de asignación comercial, es necesario comprender la estrucutura y le contenido de los datos

En esta primera etapa se revisan dimensiones, columnas, tipos de datos, y valoramos no nulos en cada dataset

In [18]:
for nombre, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"Dataset: {nombre}")
    print(f"Dimensiones: {df.shape}")
    print(f"Columnas: {df.columns.tolist()}")
    df.info()


Dataset: actividad
Dimensiones: (264, 5)
Columnas: ['id', 'registro_id', 'usuario_id', 'tipo', 'fecha']
<class 'pandas.DataFrame'>
RangeIndex: 264 entries, 0 to 263
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           264 non-null    int64
 1   registro_id  264 non-null    int64
 2   usuario_id   264 non-null    int64
 3   tipo         264 non-null    str  
 4   fecha        264 non-null    str  
dtypes: int64(3), str(2)
memory usage: 10.4 KB

Dataset: ausencias
Dimensiones: (8, 5)
Columnas: ['id', 'usuario_id', 'desde', 'hasta', 'motivo']
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          8 non-null      int64
 1   usuario_id  8 non-null      int64
 2   desde       8 non-null      str  
 3   hasta       7 non-null      str  
 4   motivo      8 non-null      str  
dtypes: int64(

In [19]:
print("=== ROLES ===")
print(usuarios["rol"].value_counts(dropna=False))

print("\n=== ZONAS ===")
print(usuarios["zona"].value_counts(dropna=False))

print("\n=== SEGMENTOS DE ESPECIALIZACIÓN ===")
print(usuarios["segmento_experto"].value_counts(dropna=False))

print("\n=== ESTADO DE ACTIVIDAD ===")
print(usuarios["activo"].value_counts(dropna=False))

print("\n=== EQUIPOS ===")
print(usuarios["equipo_id"].value_counts(dropna=False))

=== ROLES ===
rol
vendedor    13
lider        4
admin        1
Name: count, dtype: int64

=== ZONAS ===
zona
Centro       6
Antioquia    2
Occidente    2
Costa        2
centro       1
CENTRO       1
ANTIOQUIA    1
occidente    1
costa        1
NaN          1
Name: count, dtype: int64

=== SEGMENTOS DE ESPECIALIZACIÓN ===
segmento_experto
pyme           8
corporativo    6
industrial     3
NaN            1
Name: count, dtype: int64

=== ESTADO DE ACTIVIDAD ===
activo
True     17
False     1
Name: count, dtype: int64

=== EQUIPOS ===
equipo_id
1.0     4
2.0     4
3.0     3
4.0     3
5.0     2
NaN     1
99.0    1
Name: count, dtype: int64


In [20]:
print(equipos)

   id              nombre  lider_id       zona
0   1  Corporativo Centro       2.0     Centro
1   2         PYME Centro       5.0     Centro
2   3           Antioquia       9.0  Antioquia
3   4           Occidente      12.0  Occidente
4   5               Costa       NaN      Costa


In [21]:
print("=== USUARIOS SIN EQUIPO ===")
display(
    usuarios[usuarios["equipo_id"].isna()]
)

print("\n=== USUARIOS CON EQUIPO NO EXISTENTE ===")
display(
    usuarios[~usuarios["equipo_id"].isin(equipos["id"]) & usuarios["equipo_id"].notna()]
)

=== USUARIOS SIN EQUIPO ===


,id,nombre,email,rol,equipo_id,zona,segmento_experto,capacidad_maxima,fecha_ingreso,activo
0,1,Diana Marcela Ospina,diana.ospina@empresa.co,admin,NaN,Centro,NaN,NaN,2019-02-11,True



=== USUARIOS CON EQUIPO NO EXISTENTE ===


,id,nombre,email,rol,equipo_id,zona,segmento_experto,capacidad_maxima,fecha_ingreso,activo
16,17,Daniela Quintero Ríos,daniela.quintero@empresa.co,vendedor,99.0,Costa,pyme,42.0,2025-02-10,True


In [22]:
print("=== ESTADOS ===")
print(registros["estado"].value_counts(dropna=False))

print("\n=== ZONAS ===")
print(registros["zona"].value_counts(dropna=False))

print("\n=== SECTORES ===")
print(registros["sector"].value_counts(dropna=False))

print("\n=== FUENTES ===")
print(registros["fuente"].value_counts(dropna=False))

=== ESTADOS ===
estado
nuevo         67
en_gestion    46
asignado      30
descartado    20
Nuevo          4
Name: count, dtype: int64

=== ZONAS ===
zona
Centro        20
Occidente     19
Antioquia     16
Costa         15
COSTA         13
ANT           12
occidente     12
Bogotá        12
CENTRO        12
Occidente     10
NaN            8
antioquia      7
costa          6
centro         5
Name: count, dtype: int64

=== SECTORES ===
sector
Alimentos y bebidas        17
Metalmecánica              13
Manufactura                12
Textil                     12
Retail                     11
Hotelería                  11
Química                    11
Educación                  11
Salud                       9
Minería                     9
Agroindustria               8
Construcción                8
Inmobiliario                8
Transporte y logística      8
Tecnología                  8
NaN                         6
Servicios profesionales     5
Name: count, dtype: int64

=== FUENTES ===
fuen

### 5.4. Inspección de categorías y normalización potencial

Se inspeccionan las categorías originales de estado y zona para distinguir diferencias de escritura, posibles espacios adicionales y valores que podrían representar conceptos equivalentes. En esta etapa no se modifican los datos ni se consolidan categorías; las equivalencias se validarán antes de incorporarlas al motor.


In [25]:
print("=== ESTADOS ORIGINALES ===")
for valor in sorted(registros["estado"].dropna().unique()):
    print(repr(valor))

print("\n=== ZONAS ORIGINALES DE LOS REGISTROS ===")
for valor in sorted(registros["zona"].dropna().unique()):
    print(repr(valor))

print("\n=== ZONAS ORIGINALES DE LOS USUARIOS ===")
for valor in sorted(usuarios["zona"].dropna().unique()):
    print(repr(valor))

print("\n=== ZONAS ORIGINALES DE LOS EQUIPOS ===")
for valor in sorted(equipos["zona"].dropna().unique()):
    print(repr(valor))

=== ESTADOS ORIGINALES ===
'Nuevo'
'asignado'
'descartado'
'en_gestion'
'nuevo'

=== ZONAS ORIGINALES DE LOS REGISTROS ===
'ANT'
'Antioquia'
'Bogotá'
'CENTRO'
'COSTA'
'Centro'
'Costa'
'Occidente'
'Occidente '
'antioquia'
'centro'
'costa'
'occidente'

=== ZONAS ORIGINALES DE LOS USUARIOS ===
'ANTIOQUIA'
'Antioquia'
'CENTRO'
'Centro'
'Costa'
'Occidente'
'centro'
'costa'
'occidente'

=== ZONAS ORIGINALES DE LOS EQUIPOS ===
'Antioquia'
'Centro'
'Costa'
'Occidente'


### 5.6. Simulación de normalización de categorías

Se simula una normalización textual de estados y zonas mediante la eliminación de espacios externos y la conversión a minúsculas. El objetivo es medir qué categorías se consolidarían por diferencias de formato.

Esta transformación es exploratoria: no modifica los DataFrames originales ni establece equivalencias semánticas como `ANT` → `Antioquia`. Esas equivalencias requieren una decisión de negocio explícita.


In [26]:
def normalizar_texto(serie):
    return serie.str.strip().str.lower()


print("=== ESTADOS: ANTES ===")
print(registros["estado"].value_counts(dropna=False))

print("\n=== ESTADOS: NORMALIZACIÓN PROVISIONAL ===")
print(normalizar_texto(registros["estado"]).value_counts(dropna=False))


print("\n=== ZONAS DE REGISTROS: NORMALIZACIÓN PROVISIONAL ===")
print(normalizar_texto(registros["zona"]).value_counts(dropna=False))


print("\n=== ZONAS DE USUARIOS: NORMALIZACIÓN PROVISIONAL ===")
print(normalizar_texto(usuarios["zona"]).value_counts(dropna=False))


print("\n=== ZONAS DE EQUIPOS: NORMALIZACIÓN PROVISIONAL ===")
print(normalizar_texto(equipos["zona"]).value_counts(dropna=False))

=== ESTADOS: ANTES ===
estado
nuevo         67
en_gestion    46
asignado      30
descartado    20
Nuevo          4
Name: count, dtype: int64

=== ESTADOS: NORMALIZACIÓN PROVISIONAL ===
estado
nuevo         71
en_gestion    46
asignado      30
descartado    20
Name: count, dtype: int64

=== ZONAS DE REGISTROS: NORMALIZACIÓN PROVISIONAL ===
zona
occidente    41
centro       37
costa        34
antioquia    23
ant          12
bogotá       12
NaN           8
Name: count, dtype: int64

=== ZONAS DE USUARIOS: NORMALIZACIÓN PROVISIONAL ===
zona
centro       8
antioquia    3
occidente    3
costa        3
NaN          1
Name: count, dtype: int64

=== ZONAS DE EQUIPOS: NORMALIZACIÓN PROVISIONAL ===
zona
centro       2
antioquia    1
occidente    1
costa        1
Name: count, dtype: int64


## 6. Análisis de capacidad de los usuarios

La capacidad máxima representa una posible restricción del proceso de asignación. Se examinan los valores disponibles, los datos faltantes y la distribución por rol y estado de actividad.

Este análisis busca determinar qué información tenemos para evaluar la capacidad de los usuarios. No se asumirá que la capacidad máxima equivale a la carga disponible, ya que primero será necesario comprender las asignaciones y la actividad histórica.


In [28]:
print("=== CAPACIDAD MÁXIMA ===")

display(
    usuarios[
        [
            "id",
            "nombre",
            "rol",
            "equipo_id",
            "capacidad_maxima",
            "activo",
        ]
    ].sort_values("id")
)

print("\n=== RESUMEN ESTADÍSTICO DE CAPACIDAD ===")
print(usuarios["capacidad_maxima"].describe())

print("\n=== CAPACIDAD FALTANTE POR ROL ===")
display(
    usuarios.groupby("rol", dropna=False)["capacidad_maxima"]
    .agg(
        total="size",
        faltantes=lambda s: s.isna().sum()
    )
)

=== CAPACIDAD MÁXIMA ===


,id,nombre,rol,equipo_id,capacidad_maxima,activo
0,1,Diana Marcela Ospina,admin,NaN,NaN,True
1,2,Carlos Mauricio Pardo,lider,1.0,40.0,True
2,3,Juliana Ríos Gaviria,vendedor,1.0,35.0,True
3,4,Nicolás Restrepo Vélez,vendedor,1.0,35.0,False
4,5,Paula Andrea Cárdenas,lider,2.0,45.0,True
5,6,Andrés Felipe Cuéllar,vendedor,2.0,NaN,True
6,7,Laura Bermúdez Sáenz,vendedor,2.0,0.0,True
7,8,Santiago Mejía Botero,vendedor,2.0,45.0,True
8,9,Mónica Álvarez Uribe,lider,3.0,40.0,True
9,10,Jorge Iván Arango,vendedor,3.0,40.0,True



=== RESUMEN ESTADÍSTICO DE CAPACIDAD ===
count    16.00000
mean     37.18750
std      10.40012
min       0.00000
25%      37.25000
50%      40.00000
75%      42.00000
max      45.00000
Name: capacidad_maxima, dtype: float64

=== CAPACIDAD FALTANTE POR ROL ===


,total,faltantes
rol,,
admin,1,1
lider,4,0
vendedor,13,1


In [29]:
print("=== USUARIOS SIN CAPACIDAD DEFINIDA ===")

display(
    usuarios.loc[
        usuarios["capacidad_maxima"].isna(),
        [
            "id",
            "nombre",
            "rol",
            "equipo_id",
            "zona",
            "segmento_experto",
            "capacidad_maxima",
            "activo",
        ],
    ]
)

=== USUARIOS SIN CAPACIDAD DEFINIDA ===


,id,nombre,rol,equipo_id,zona,segmento_experto,capacidad_maxima,activo
0,1,Diana Marcela Ospina,admin,NaN,Centro,NaN,NaN,True
5,6,Andrés Felipe Cuéllar,vendedor,2.0,CENTRO,pyme,NaN,True


## 7. Análisis de actividad histórica

El dataset `actividad` contiene el historial de acciones realizadas sobre
los registros comerciales.

Antes de utilizar esta información para estimar carga de trabajo o
balancear asignaciones, es necesario conocer los tipos de actividad
existentes y su distribución.

No se asumirá que una actividad representa necesariamente una asignación
actual ni que todas las actividades tienen el mismo peso.

In [30]:
print("=== TIPOS DE ACTIVIDAD ===")
print(actividad["tipo"].value_counts(dropna=False))

=== TIPOS DE ACTIVIDAD ===
tipo
llamada          66
correo           51
cambio_estado    36
nota             32
whatsapp         31
reunión          26
cotización       22
Name: count, dtype: int64


### 7.1. Distribución de actividad por usuario

Se analiza la cantidad de actividades asociadas a cada usuario para
observar la distribución histórica de interacciones.

Este indicador no se interpreta todavía como carga actual, ya que una
actividad puede corresponder a un registro que ya fue cerrado,
descartado o reasignado.

In [31]:
actividad_por_usuario = (
    actividad.groupby("usuario_id")
    .size()
    .reset_index(name="cantidad_actividades")
    .sort_values("cantidad_actividades", ascending=False)
)

display(actividad_por_usuario)

,usuario_id,cantidad_actividades
4,8,40
8,14,34
1,4,31
12,18,31
5,10,30
9,15,26
0,3,21
7,13,16
11,17,15
10,16,8


### 7.2. Actividad histórica y estado actual de los registros

Se cruza la actividad histórica con el estado actual de cada registro
comercial.

El objetivo es determinar si las actividades registradas corresponden
a registros actualmente asignados, en gestión, descartados o nuevos.

Este análisis permitirá evaluar si la actividad puede utilizarse como
aproximación de carga histórica o si se requiere una fuente adicional
para determinar la carga actual.

In [32]:
actividad_estado = actividad.merge(
    registros[["id", "estado"]],
    left_on="registro_id",
    right_on="id",
    how="left",
    suffixes=("_actividad", "_registro")
)

print("=== ACTIVIDADES POR ESTADO ACTUAL DEL REGISTRO ===")

display(
    actividad_estado["estado"]
    .value_counts(dropna=False)
)

=== ACTIVIDADES POR ESTADO ACTUAL DEL REGISTRO ===


estado
en_gestion    121
asignado       86
descartado     57
Name: count, dtype: int64

### 7.3. Intensidad de actividad por registro

Se analiza la cantidad de actividades históricas asociadas a cada registro
comercial y su relación con el estado actual.

Este análisis permite identificar si existen registros con múltiples
interacciones y diferenciar la actividad histórica de una posible medida
de carga actual.

La actividad histórica no se utilizará directamente como carga actual
hasta validar que existe una relación confiable entre las actividades y
la asignación vigente de cada registro.

In [34]:
actividad_por_registro = (
    actividad
    .groupby("registro_id")
    .size()
    .reset_index(name="cantidad_actividades")
    .merge(
        registros[["id", "razon_social", "estado", "zona"]],
        left_on="registro_id",
        right_on="id",
        how="left"
    )
)

print("=== ACTIVIDADES POR REGISTRO ===")

display(
    actividad_por_registro[
        [
            "registro_id",
            "razon_social",
            "estado",
            "zona",
            "cantidad_actividades",
        ]
    ]
    .sort_values("cantidad_actividades", ascending=False)
    .head(20)
)

print("\n=== RESUMEN DE ACTIVIDADES POR REGISTRO ===")

print(
    actividad_por_registro["cantidad_actividades"]
    .describe()
)

=== ACTIVIDADES POR REGISTRO ===


,registro_id,razon_social,estado,zona,cantidad_actividades
1,70,Transportes Santa Bárbara S.A.S.,asignado,ANT,5
6,75,Soluciones Yumbo S.A.S.,asignado,Centro,5
10,79,Transportes Monserrate S.A.S.,en_gestion,occidente,5
11,80,Alimentos Orinoquía S.A.S.,asignado,Centro,5
14,83,Suministros Zipaquirá S.A.S.,en_gestion,ANT,5
28,97,Comercializadora Los Nevados S.A.S.,en_gestion,ANT,5
88,157,Logística La Candelaria S.A.S.,en_gestion,CENTRO,5
94,163,Distribuidora El Dorado Ltda.,descartado,Costa,5
69,138,Suministros El Dorado S. en C.,en_gestion,ANT,5
68,137,Suministros Tequendama S.A.S.,descartado,COSTA,5



=== RESUMEN DE ACTIVIDADES POR REGISTRO ===
count    96.000000
mean      2.750000
std       1.486784
min       1.000000
25%       1.000000
50%       3.000000
75%       4.000000
max       5.000000
Name: cantidad_actividades, dtype: float64


### 7.4. Actividad histórica por usuario y estado del registro

Se analiza la distribución de actividades históricas por usuario y por
estado actual del registro.

El objetivo es identificar patrones de carga histórica y determinar si la
actividad puede aportar información complementaria para el balance de
asignaciones.

Este indicador se considera histórico y no se interpreta como la carga
actual del vendedor, ya que los datos no contienen explícitamente una
relación de propiedad vigente entre usuario y registro.

In [35]:
actividad_usuario_estado = (
    actividad_estado
    .groupby(["usuario_id", "estado"])
    .size()
    .reset_index(name="cantidad_actividades")
)

print("=== ACTIVIDAD POR USUARIO Y ESTADO DEL REGISTRO ===")

display(
    actividad_usuario_estado
    .sort_values(
        ["usuario_id", "cantidad_actividades"],
        ascending=[True, False]
    )
)

=== ACTIVIDAD POR USUARIO Y ESTADO DEL REGISTRO ===


,usuario_id,estado,cantidad_actividades
2,3,en_gestion,12
1,3,descartado,5
0,3,asignado,4
3,4,asignado,14
5,4,en_gestion,9
4,4,descartado,8
6,6,en_gestion,7
7,7,en_gestion,4
10,8,en_gestion,25
9,8,descartado,10


## 8. Análisis de ausencias y disponibilidad

Las ausencias representan una restricción temporal para la asignación de
registros.

Se revisan las fechas de inicio y finalización de cada ausencia, así como
los motivos registrados. También se identifican ausencias sin fecha de
finalización, ya que requieren un tratamiento explícito dentro del motor.

Esta información permitirá definir la disponibilidad de los usuarios como
una restricción del proceso de asignación.

In [36]:
print("=== AUSENCIAS ===")

display(ausencias)

print("\n=== MOTIVOS ===")
print(
    ausencias["motivo"]
    .value_counts(dropna=False)
)

print("\n=== AUSENCIAS SIN FECHA DE FINALIZACIÓN ===")

display(
    ausencias[ausencias["hasta"].isna()]
)

print("\n=== FECHAS ===")

print("Desde:")
print(ausencias["desde"].min(), "→", ausencias["desde"].max())

print("\nHasta:")
print(ausencias["hasta"].min(), "→", ausencias["hasta"].max())

=== AUSENCIAS ===


,id,usuario_id,desde,hasta,motivo
0,1,3,2026-09-14,2026-09-25,Vacaciones
1,2,10,2026-09-16,2026-09-18,Incapacidad
2,3,13,2026-08-03,2026-08-17,Vacaciones
3,4,16,2026-09-21,2026-10-02,Vacaciones
4,5,8,2026-07-01,2026-07-15,Licencia no remunerada
5,6,11,2026-09-15,NaN,Incapacidad
6,7,15,2026-06-10,2026-06-12,Calamidad doméstica
7,8,6,2026-09-28,2026-10-05,Vacaciones



=== MOTIVOS ===
motivo
Vacaciones                4
Incapacidad               2
Licencia no remunerada    1
Calamidad doméstica       1
Name: count, dtype: int64

=== AUSENCIAS SIN FECHA DE FINALIZACIÓN ===


,id,usuario_id,desde,hasta,motivo
5,6,11,2026-09-15,NaN,Incapacidad



=== FECHAS ===
Desde:
2026-06-10 → 2026-09-28

Hasta:
2026-06-12 → 2026-10-05


In [37]:
print("=== DETALLE DE TODAS LAS AUSENCIAS ===")

display(
    ausencias.sort_values(["usuario_id", "desde"])
)

print("\n=== AUSENCIAS SIN FECHA DE FINALIZACIÓN ===")

display(
    ausencias[ausencias["hasta"].isna()]
)

print("\n=== AUSENCIAS POR USUARIO ===")

print(
    ausencias["usuario_id"]
    .value_counts()
    .sort_index()
)

=== DETALLE DE TODAS LAS AUSENCIAS ===


,id,usuario_id,desde,hasta,motivo
0,1,3,2026-09-14,2026-09-25,Vacaciones
7,8,6,2026-09-28,2026-10-05,Vacaciones
4,5,8,2026-07-01,2026-07-15,Licencia no remunerada
1,2,10,2026-09-16,2026-09-18,Incapacidad
5,6,11,2026-09-15,NaN,Incapacidad
2,3,13,2026-08-03,2026-08-17,Vacaciones
6,7,15,2026-06-10,2026-06-12,Calamidad doméstica
3,4,16,2026-09-21,2026-10-02,Vacaciones



=== AUSENCIAS SIN FECHA DE FINALIZACIÓN ===


,id,usuario_id,desde,hasta,motivo
5,6,11,2026-09-15,NaN,Incapacidad



=== AUSENCIAS POR USUARIO ===
usuario_id
3     1
6     1
8     1
10    1
11    1
13    1
15    1
16    1
Name: count, dtype: int64


In [38]:
print("=== DETALLE DE AUSENCIAS ===")

display(
    ausencias[
        ["usuario_id", "desde", "hasta", "motivo"]
    ].sort_values("desde")
)

=== DETALLE DE AUSENCIAS ===


,usuario_id,desde,hasta,motivo
6,15,2026-06-10,2026-06-12,Calamidad doméstica
4,8,2026-07-01,2026-07-15,Licencia no remunerada
2,13,2026-08-03,2026-08-17,Vacaciones
0,3,2026-09-14,2026-09-25,Vacaciones
5,11,2026-09-15,NaN,Incapacidad
1,10,2026-09-16,2026-09-18,Incapacidad
3,16,2026-09-21,2026-10-02,Vacaciones
7,6,2026-09-28,2026-10-05,Vacaciones


## 9. Identificación de registros candidatos a asignación

El reto requiere identificar los registros pendientes que serán procesados
por el motor.

El dataset contiene diferentes estados (`nuevo`, `en_gestion`, `asignado` y
`descartado`). Antes de definir cuáles ingresan al motor, se analiza su
relación con la actividad histórica.

La decisión sobre los estados candidatos debe basarse en la información
disponible y quedar documentada explícitamente, evitando asumir una
semántica que no esté respaldada por los datos.

In [39]:
estado_actividad = (
    actividad
    .groupby("registro_id")
    .size()
    .reset_index(name="cantidad_actividades")
)

registros_estado = registros.merge(
    estado_actividad,
    left_on="id",
    right_on="registro_id",
    how="left"
)

registros_estado["cantidad_actividades"] = (
    registros_estado["cantidad_actividades"]
    .fillna(0)
    .astype(int)
)

print("=== ESTADO ACTUAL VS ACTIVIDAD HISTÓRICA ===")

display(
    registros_estado
    .groupby("estado")["cantidad_actividades"]
    .agg(
        registros="size",
        con_actividad=lambda s: (s > 0).sum(),
        sin_actividad=lambda s: (s == 0).sum(),
        promedio="mean",
        maximo="max"
    )
)

=== ESTADO ACTUAL VS ACTIVIDAD HISTÓRICA ===


,registros,con_actividad,sin_actividad,promedio,maximo
estado,,,,,
Nuevo,4,0,4,0.000000,0
asignado,30,30,0,2.866667,5
descartado,20,20,0,2.850000,5
en_gestion,46,46,0,2.630435,5
nuevo,67,0,67,0.000000,0


## 10. Perfil de los registros candidatos

Se analizan los registros cuyo estado corresponde a `nuevo`, considerados
como candidatos al proceso de asignación.

El objetivo es conocer las características de los registros que ingresarán
al motor y determinar qué variables pueden utilizarse como criterios de
asignación.

Se revisan variables geográficas, sectoriales, tamaño de empresa,
ingresos estimados y fuente de adquisición, además de los valores faltantes.

In [40]:
registros_candidatos = registros[
    normalizar_texto(registros["estado"]) == "nuevo"
].copy()

print("=== REGISTROS CANDIDATOS ===")
print("Cantidad:", len(registros_candidatos))

print("\n=== ZONAS ===")
print(
    registros_candidatos["zona"]
    .value_counts(dropna=False)
)

print("\n=== SECTORES ===")
print(
    registros_candidatos["sector"]
    .value_counts(dropna=False)
)

print("\n=== FUENTES ===")
print(
    registros_candidatos["fuente"]
    .value_counts(dropna=False)
)

print("\n=== VARIABLES NUMÉRICAS ===")

display(
    registros_candidatos[
        ["empleados", "ingresos_estimados"]
    ].describe()
)

print("\n=== VALORES FALTANTES ===")

display(
    registros_candidatos.isna().sum()
    .sort_values(ascending=False)
)

=== REGISTROS CANDIDATOS ===
Cantidad: 71

=== ZONAS ===
zona
Occidente     11
Centro         7
Costa          6
Occidente      6
ANT            6
NaN            5
Antioquia      5
occidente      5
Bogotá         5
COSTA          4
costa          3
centro         3
CENTRO         3
antioquia      2
Name: count, dtype: int64

=== SECTORES ===
sector
Retail                     8
Alimentos y bebidas        7
Textil                     7
Química                    6
Hotelería                  5
Minería                    5
Metalmecánica              4
Manufactura                4
Servicios profesionales    4
Construcción               4
Salud                      3
NaN                        3
Agroindustria              3
Inmobiliario               2
Transporte y logística     2
Tecnología                 2
Educación                  2
Name: count, dtype: int64

=== FUENTES ===
fuente
referido    17
lista       17
evento      13
campaña     12
web         12
Name: count, dtype: int64

=== 

,empleados,ingresos_estimados
count,66.000000,6.800000e+01
mean,324.090909,1.837282e+10
std,467.570076,2.992792e+10
min,8.000000,3.350000e+08
25%,25.000000,1.966500e+09
50%,80.000000,4.943500e+09
75%,420.000000,1.682950e+10
max,2100.000000,1.195990e+11



=== VALORES FALTANTES ===


notas                 11
empleados              5
zona                   5
sector                 3
ingresos_estimados     3
nit                    0
id                     0
razon_social           0
ciudad                 0
fuente                 0
estado                 0
fecha_creacion         0
dtype: int64

## 11. Relación entre sector de los registros y especialización de los usuarios

Se compara la distribución de sectores presentes en los registros
candidatos con los segmentos de especialización declarados por los
usuarios.

El objetivo es determinar si existe una relación observable entre ambas
variables que pueda utilizarse como criterio de asignación.

No se asumirá una equivalencia directa entre `sector` y
`segmento_experto`; cualquier correspondencia deberá definirse
explícitamente en la lógica del motor.

In [41]:
print("=== SECTORES DE LOS CANDIDATOS ===")

display(
    registros_candidatos["sector"]
    .value_counts(dropna=False)
)

print("\n=== ESPECIALIZACIONES DE LOS USUARIOS ===")

display(
    usuarios["segmento_experto"]
    .value_counts(dropna=False)
)

print("\n=== EJEMPLOS DE REGISTROS CANDIDATOS ===")

display(
    registros_candidatos[
        [
            "id",
            "razon_social",
            "sector",
            "empleados",
            "ingresos_estimados",
            "ciudad",
            "zona",
            "fuente",
        ]
    ].head(15)
)

=== SECTORES DE LOS CANDIDATOS ===


sector
Retail                     8
Alimentos y bebidas        7
Textil                     7
Química                    6
Hotelería                  5
Minería                    5
Metalmecánica              4
Manufactura                4
Servicios profesionales    4
Construcción               4
Salud                      3
NaN                        3
Agroindustria              3
Inmobiliario               2
Transporte y logística     2
Tecnología                 2
Educación                  2
Name: count, dtype: int64


=== ESPECIALIZACIONES DE LOS USUARIOS ===


segmento_experto
pyme           8
corporativo    6
industrial     3
NaN            1
Name: count, dtype: int64


=== EJEMPLOS DE REGISTROS CANDIDATOS ===


,id,razon_social,sector,empleados,ingresos_estimados,ciudad,zona,fuente
0,1,Servicios Girardot S.A.,Metalmecánica,8.0,1.866000e+09,Montería,Costa,campaña
1,2,COMERCIALIZADORA ANDINA SAS,Salud,40.0,2.156000e+09,Palmira,Occidente,web
2,3,Industrias Andina S.A.S.,Alimentos y bebidas,260.0,7.418000e+09,Manizales,NaN,web
3,4,Constructora Caribe Azul Ltda.,Retail,40.0,1.967000e+09,Manizales,Occidente,web
4,5,Distribuidora La Esperanza Ltda.,Hotelería,260.0,1.059100e+10,Envigado,Antioquia,campaña
5,6,Suministros Vallecaucana Ltda.,Metalmecánica,NaN,3.350000e+08,Montería,COSTA,campaña
6,7,Grupo El Poblado S.A.,Manufactura,18.0,2.186000e+09,Cali,Occidente,evento
7,8,Manufacturas El Dorado S.A.,NaN,1400.0,5.272200e+10,Medellín,Antioquia,referido
8,9,Soluciones Yumbo S.A.S.,Química,120.0,3.432000e+09,Envigado,ANT,web
9,10,Textiles La Esperanza S.A.,Alimentos y bebidas,650.0,NaN,Envigado,antioquia,evento


In [42]:
print("=== DISTRIBUCIÓN DE SECTORES EN LOS REGISTROS CANDIDATOS ===")

display(
    registros_candidatos["sector"]
    .value_counts(dropna=False)
)

print("\n=== DISTRIBUCIÓN DE SEGMENTOS DE LOS USUARIOS ===")

display(
    usuarios["segmento_experto"]
    .value_counts(dropna=False)
)

=== DISTRIBUCIÓN DE SECTORES EN LOS REGISTROS CANDIDATOS ===


sector
Retail                     8
Alimentos y bebidas        7
Textil                     7
Química                    6
Hotelería                  5
Minería                    5
Metalmecánica              4
Manufactura                4
Servicios profesionales    4
Construcción               4
Salud                      3
NaN                        3
Agroindustria              3
Inmobiliario               2
Transporte y logística     2
Tecnología                 2
Educación                  2
Name: count, dtype: int64


=== DISTRIBUCIÓN DE SEGMENTOS DE LOS USUARIOS ===


segmento_experto
pyme           8
corporativo    6
industrial     3
NaN            1
Name: count, dtype: int64

In [43]:
print("=== EMPLEADOS DE LOS REGISTROS CANDIDATOS ===")

display(
    registros_candidatos["empleados"].describe()
)

print("\n=== INGRESOS ESTIMADOS DE LOS REGISTROS CANDIDATOS ===")

display(
    registros_candidatos["ingresos_estimados"].describe()
)

=== EMPLEADOS DE LOS REGISTROS CANDIDATOS ===


count      66.000000
mean      324.090909
std       467.570076
min         8.000000
25%        25.000000
50%        80.000000
75%       420.000000
max      2100.000000
Name: empleados, dtype: float64


=== INGRESOS ESTIMADOS DE LOS REGISTROS CANDIDATOS ===


count    6.800000e+01
mean     1.837282e+10
std      2.992792e+10
min      3.350000e+08
25%      1.966500e+09
50%      4.943500e+09
75%      1.682950e+10
max      1.195990e+11
Name: ingresos_estimados, dtype: float64

In [44]:
print("=== EMPLEADOS DE LOS REGISTROS CANDIDATOS ===")

display(
    registros_candidatos["empleados"].describe()
)

print("\n=== REGISTROS SIN EMPLEADOS ===")

display(
    registros_candidatos[
        registros_candidatos["empleados"].isna()
    ][
        [
            "id",
            "razon_social",
            "sector",
            "empleados",
            "ingresos_estimados",
            "zona",
        ]
    ]
)

=== EMPLEADOS DE LOS REGISTROS CANDIDATOS ===


count      66.000000
mean      324.090909
std       467.570076
min         8.000000
25%        25.000000
50%        80.000000
75%       420.000000
max      2100.000000
Name: empleados, dtype: float64


=== REGISTROS SIN EMPLEADOS ===


,id,razon_social,sector,empleados,ingresos_estimados,zona
5,6,Suministros Vallecaucana Ltda.,Metalmecánica,NaN,3.350000e+08,COSTA
14,15,Soluciones Monserrate S.A.S.,Construcción,NaN,1.973000e+09,Occidente
31,32,Suministros Cafetera Ltda.,Construcción,NaN,1.711000e+10,occidente
46,47,Soluciones Orinoquía S.A.S.,Textil,NaN,2.311000e+09,Occidente
58,59,Servicios San Rafael S.A.,Tecnología,NaN,8.220000e+08,CENTRO


In [45]:
print("=== EMPLEADOS E INGRESOS DE LOS REGISTROS CANDIDATOS ===")

display(
    registros_candidatos[
        [
            "id",
            "razon_social",
            "sector",
            "empleados",
            "ingresos_estimados",
            "zona",
        ]
    ]
    .sort_values(
        ["ingresos_estimados", "empleados"],
        ascending=[False, False]
    )
)

=== EMPLEADOS E INGRESOS DE LOS REGISTROS CANDIDATOS ===


,id,razon_social,sector,empleados,ingresos_estimados,zona
29,30,Logística El Poblado S.A.S.,Hotelería,2100.0,1.195990e+11,Occidente
50,51,Manufacturas Bolívar S.A.,Textil,1400.0,1.186620e+11,Antioquia
48,49,Servicios Zipaquirá S.A.,Educación,420.0,1.106020e+11,costa
65,66,Agropecuaria Caribe Azul Ltda.,Transporte y logística,900.0,1.082260e+11,ANT
27,28,Soluciones Las Palmas Ltda.,Textil,1400.0,7.105500e+10,COSTA
...,...,...,...,...,...,...
56,57,Manufacturas Caribe Azul S.A.S.,Química,18.0,4.100000e+08,Occidente
5,6,Suministros Vallecaucana Ltda.,Metalmecánica,NaN,3.350000e+08,COSTA
66,67,Manufacturas Las Palmas S. en C.,Retail,1400.0,NaN,Occidente
24,25,Agropecuaria La Candelaria S.A.S.,Química,900.0,NaN,Costa


In [46]:
print("=== CAPACIDAD TOTAL DE USUARIOS ACTIVOS ===")

capacidad_activos = usuarios.loc[
    usuarios["activo"] == True,
    "capacidad_maxima"
].sum()

print("Capacidad total:", capacidad_activos)
print("Registros candidatos:", len(registros_candidatos))

print("\n=== CAPACIDAD POR ROL ===")

display(
    usuarios[
        [
            "id",
            "nombre",
            "rol",
            "capacidad_maxima",
            "activo"
        ]
    ]
    .sort_values("capacidad_maxima", ascending=False)
)

=== CAPACIDAD TOTAL DE USUARIOS ACTIVOS ===
Capacidad total: 560.0
Registros candidatos: 71

=== CAPACIDAD POR ROL ===


,id,nombre,rol,capacidad_maxima,activo
4,5,Paula Andrea Cárdenas,lider,45.0,True
7,8,Santiago Mejía Botero,vendedor,45.0,True
16,17,Daniela Quintero Ríos,vendedor,42.0,True
15,16,Julián Camargo Peña,vendedor,42.0,True
14,15,Marcela Fonseca Prieto,vendedor,42.0,True
9,10,Jorge Iván Arango,vendedor,40.0,True
8,9,Mónica Álvarez Uribe,lider,40.0,True
1,2,Carlos Mauricio Pardo,lider,40.0,True
10,11,Valentina Zapata Ruiz,vendedor,40.0,True
13,14,Óscar Hernán Salazar,vendedor,38.0,True


In [47]:
print("=== REGISTROS CON ACTIVIDAD ===")

registros_con_actividad = (
    actividad["registro_id"]
    .nunique()
)

print(
    "Registros con actividad:",
    registros_con_actividad
)

print(
    "Registros candidatos:",
    len(registros_candidatos)
)

print("\n=== ÚLTIMA ACTIVIDAD POR REGISTRO ===")

ultima_actividad = (
    actividad
    .sort_values("fecha")
    .groupby("registro_id")
    .tail(1)
)

display(
    ultima_actividad[
        [
            "registro_id",
            "usuario_id",
            "tipo",
            "fecha"
        ]
    ].head(20)
)

=== REGISTROS CON ACTIVIDAD ===
Registros con actividad: 96
Registros candidatos: 71

=== ÚLTIMA ACTIVIDAD POR REGISTRO ===


,registro_id,usuario_id,tipo,fecha
0,134,18,cotización,2026-02-26
5,102,13,correo,2026-03-12
10,121,13,correo,2026-03-18
11,132,10,nota,2026-03-18
21,93,4,correo,2026-03-29
22,96,4,correo,2026-03-29
24,131,15,cotización,2026-03-30
25,95,18,correo,2026-03-31
26,156,13,whatsapp,2026-03-31
28,128,18,llamada,2026-04-01


## 12. Reconstrucción de carga histórica

La tabla `actividad` contiene el usuario que realizó cada actividad
sobre un registro, pero no contiene explícitamente una relación de
asignación vigente.

Por esta razón, `actividad.usuario_id` no será utilizado como fuente
de verdad para determinar la carga actual de un usuario.

La actividad histórica podrá utilizarse como información contextual y
para trazabilidad, pero la carga actual será gestionada por el propio
motor mediante una entidad explícita de asignación.

Esto evita asumir que el usuario que realizó la última actividad es
necesariamente el responsable actual del registro.

## 13. Fecha de evaluación del motor

Para garantizar que las decisiones del motor sean reproducibles, se
define explícitamente una fecha de evaluación.

Para este análisis se utilizará:

**Fecha de evaluación: 2026-10-03**

Esta fecha representa el momento en el que el motor consulta la
disponibilidad de los usuarios y ejecuta una asignación.

La fecha se utilizará principalmente para evaluar las ausencias
registradas. Un usuario se considerará no disponible cuando la fecha
de evaluación se encuentre dentro de su período de ausencia.

La fecha no se obtendrá directamente del reloj del sistema, ya que
hacerlo produciría resultados diferentes dependiendo del momento en
que se ejecute el motor.

En una implementación productiva, esta fecha debería recibirse como
un parámetro de la ejecución o generarse explícitamente al momento de
crear la asignación y almacenarse en la trazabilidad.

In [48]:
from datetime import date

FECHA_EVALUACION = date(2026, 10, 3)

print("Fecha de evaluación del motor:", FECHA_EVALUACION)

Fecha de evaluación del motor: 2026-10-03


In [49]:
ausencias["desde"] = pd.to_datetime(ausencias["desde"])
ausencias["hasta"] = pd.to_datetime(ausencias["hasta"])

fecha_evaluacion = pd.Timestamp(FECHA_EVALUACION)

ausencias_vigentes = ausencias[
    (ausencias["desde"] <= fecha_evaluacion)
    &
    (
        ausencias["hasta"].isna()
        |
        (ausencias["hasta"] >= fecha_evaluacion)
    )
]

print("=== AUSENCIAS VIGENTES ===")

display(ausencias_vigentes)

=== AUSENCIAS VIGENTES ===


,id,usuario_id,desde,hasta,motivo
5,6,11,2026-09-15,NaT,Incapacidad
7,8,6,2026-09-28,2026-10-05,Vacaciones


### Resultado de disponibilidad

Para la fecha de evaluación `2026-10-03`, se identifican dos usuarios
con ausencia vigente:

- Usuario 11: incapacidad iniciada el 2026-09-15 sin fecha de
  finalización registrada.
- Usuario 6: vacaciones desde el 2026-09-28 hasta el 2026-10-05.

Ambos usuarios serán excluidos de las asignaciones realizadas en esta
ejecución del motor.

La ausencia con fecha de finalización nula se considera vigente hasta
que exista explícitamente una fecha de finalización.

In [50]:
# Copia de trabajo para no modificar el dataset original
usuarios_elegibilidad = usuarios.copy()

# Fecha de evaluación
fecha_evaluacion = pd.Timestamp(FECHA_EVALUACION)

# IDs con ausencia vigente
usuarios_ausentes = set(
    ausencias_vigentes["usuario_id"]
)

# Evaluación de cada condición
usuarios_elegibilidad["cumple_activo"] = (
    usuarios_elegibilidad["activo"] == True
)

usuarios_elegibilidad["cumple_rol"] = (
    usuarios_elegibilidad["rol"].isin(["vendedor", "lider"])
)

usuarios_elegibilidad["cumple_ausencia"] = (
    ~usuarios_elegibilidad["id"].isin(usuarios_ausentes)
)

usuarios_elegibilidad["cumple_capacidad"] = (
    usuarios_elegibilidad["capacidad_maxima"].notna()
    &
    (usuarios_elegibilidad["capacidad_maxima"] > 0)
)

# Elegibilidad final
usuarios_elegibilidad["elegible"] = (
    usuarios_elegibilidad["cumple_activo"]
    &
    usuarios_elegibilidad["cumple_rol"]
    &
    usuarios_elegibilidad["cumple_ausencia"]
    &
    usuarios_elegibilidad["cumple_capacidad"]
)

# Motivo de exclusión
def determinar_motivo(row):

    if not row["cumple_activo"]:
        return "Usuario inactivo"

    if not row["cumple_rol"]:
        return "Rol no asignable"

    if not row["cumple_ausencia"]:
        return "Ausencia vigente"

    if not row["cumple_capacidad"]:
        if pd.isna(row["capacidad_maxima"]):
            return "Capacidad no definida"

        if row["capacidad_maxima"] <= 0:
            return "Capacidad no disponible"

    return "Elegible"


usuarios_elegibilidad["motivo_elegibilidad"] = (
    usuarios_elegibilidad.apply(
        determinar_motivo,
        axis=1
    )
)

display(
    usuarios_elegibilidad[
        [
            "id",
            "nombre",
            "rol",
            "capacidad_maxima",
            "activo",
            "elegible",
            "motivo_elegibilidad",
        ]
    ].sort_values("id")
)

,id,nombre,rol,capacidad_maxima,activo,elegible,motivo_elegibilidad
0,1,Diana Marcela Ospina,admin,NaN,True,False,Rol no asignable
1,2,Carlos Mauricio Pardo,lider,40.0,True,True,Elegible
2,3,Juliana Ríos Gaviria,vendedor,35.0,True,True,Elegible
3,4,Nicolás Restrepo Vélez,vendedor,35.0,False,False,Usuario inactivo
4,5,Paula Andrea Cárdenas,lider,45.0,True,True,Elegible
5,6,Andrés Felipe Cuéllar,vendedor,NaN,True,False,Ausencia vigente
6,7,Laura Bermúdez Sáenz,vendedor,0.0,True,False,Capacidad no disponible
7,8,Santiago Mejía Botero,vendedor,45.0,True,True,Elegible
8,9,Mónica Álvarez Uribe,lider,40.0,True,True,Elegible
9,10,Jorge Iván Arango,vendedor,40.0,True,True,Elegible


In [51]:
print("=== RESUMEN DE ELEGIBILIDAD ===")

print(
    usuarios_elegibilidad["elegible"]
    .value_counts()
)

print("\n=== MOTIVOS ===")

display(
    usuarios_elegibilidad["motivo_elegibilidad"]
    .value_counts()
)

=== RESUMEN DE ELEGIBILIDAD ===
elegible
True     13
False     5
Name: count, dtype: int64

=== MOTIVOS ===


motivo_elegibilidad
Elegible                   13
Ausencia vigente            2
Rol no asignable            1
Usuario inactivo            1
Capacidad no disponible     1
Name: count, dtype: int64

In [52]:
usuarios_operativos = usuarios_elegibilidad[
    usuarios_elegibilidad["elegible"]
].copy()

usuarios_operativos["carga_actual"] = 0

usuarios_operativos["capacidad_disponible"] = (
    usuarios_operativos["capacidad_maxima"]
    - usuarios_operativos["carga_actual"]
)

display(
    usuarios_operativos[
        [
            "id",
            "nombre",
            "rol",
            "capacidad_maxima",
            "carga_actual",
            "capacidad_disponible",
            "zona",
            "segmento_experto",
        ]
    ]
    .sort_values("id")
)

,id,nombre,rol,capacidad_maxima,carga_actual,capacidad_disponible,zona,segmento_experto
1,2,Carlos Mauricio Pardo,lider,40.0,0,40.0,Centro,corporativo
2,3,Juliana Ríos Gaviria,vendedor,35.0,0,35.0,Centro,corporativo
4,5,Paula Andrea Cárdenas,lider,45.0,0,45.0,Centro,pyme
7,8,Santiago Mejía Botero,vendedor,45.0,0,45.0,Centro,pyme
8,9,Mónica Álvarez Uribe,lider,40.0,0,40.0,Antioquia,industrial
9,10,Jorge Iván Arango,vendedor,40.0,0,40.0,ANTIOQUIA,industrial
11,12,Ricardo Lozano Pineda,lider,38.0,0,38.0,Occidente,corporativo
12,13,Catalina Vargas Nieto,vendedor,38.0,0,38.0,Occidente,pyme
13,14,Óscar Hernán Salazar,vendedor,38.0,0,38.0,occidente,industrial
14,15,Marcela Fonseca Prieto,vendedor,42.0,0,42.0,Costa,pyme


## 14. Compatibilidad entre zonas de registros y usuarios

In [53]:
registros_nuevos = registros[
    normalizar_texto(registros["estado"]) == "nuevo"
].copy()

registros_nuevos["zona_normalizada"] = normalizar_texto(
    registros_nuevos["zona"]
)

usuarios_operativos["zona_normalizada"] = normalizar_texto(
    usuarios_operativos["zona"]
)

print("=== ZONAS DE REGISTROS NUEVOS ===")
print(
    registros_nuevos["zona_normalizada"]
    .value_counts(dropna=False)
)

print("\n=== ZONAS DE USUARIOS ELEGIBLES ===")
print(
    usuarios_operativos["zona_normalizada"]
    .value_counts(dropna=False)
)

=== ZONAS DE REGISTROS NUEVOS ===
zona_normalizada
occidente    22
costa        13
centro       13
antioquia     7
ant           6
NaN           5
bogotá        5
Name: count, dtype: int64

=== ZONAS DE USUARIOS ELEGIBLES ===
zona_normalizada
centro       4
occidente    3
costa        3
antioquia    2
NaN          1
Name: count, dtype: int64


In [54]:
def contar_candidatos_por_zona(registro):
    zona = registro["zona_normalizada"]

    if pd.isna(zona):
        return usuarios_operativos["id"].nunique()

    return (
        usuarios_operativos["zona_normalizada"] == zona
    ).sum()


registros_nuevos["candidatos_misma_zona"] = (
    registros_nuevos.apply(
        contar_candidatos_por_zona,
        axis=1
    )
)

display(
    registros_nuevos[
        [
            "id",
            "razon_social",
            "zona",
            "zona_normalizada",
            "candidatos_misma_zona",
        ]
    ].sort_values("candidatos_misma_zona")
)

,id,razon_social,zona,zona_normalizada,candidatos_misma_zona
13,14,Textiles La Candelaria S. en C.,ANT,ant,0
8,9,Soluciones Yumbo S.A.S.,ANT,ant,0
54,55,Manufacturas Pacífico Sur S.A.,Bogotá,bogotá,0
47,48,Comercializadora Magdalena S.A.S.,ANT,ant,0
41,42,Soluciones Chicamocha S. en C.,Bogotá,bogotá,0
...,...,...,...,...,...
61,62,Textiles Pacífico Sur S. en C.,NaN,NaN,13
26,27,Industrias Orinoquía S.A.S.,NaN,NaN,13
2,3,Industrias Andina S.A.S.,NaN,NaN,13
40,41,Manufacturas Andina S. en C.,NaN,NaN,13


## 15. Diseño de la metodología Weighted Rules

La metodología **Weighted Rules** asigna cada registro nuevo al usuario elegible
con el mayor puntaje, considerando restricciones operativas y criterios
ponderados.

El proceso se divide en dos etapas:

1. Aplicar restricciones duras para determinar qué usuarios pueden recibir una
   asignación.
2. Calcular un puntaje para los candidatos válidos y seleccionar el de mayor
   puntuación.

### 15.1 Restricciones duras

Un usuario no puede participar en la asignación cuando:

- Se encuentra inactivo.
- Su rol no es asignable (`vendedor` o `lider`).
- Tiene una ausencia vigente en la fecha de evaluación.
- No tiene capacidad máxima definida.
- No tiene capacidad disponible.

Estas condiciones no generan puntuación. Funcionan como filtros de elegibilidad.

### 15.2 Criterios ponderados

Para los usuarios elegibles se consideran inicialmente dos criterios:

| Criterio | Peso máximo |
|---|---:|
| Compatibilidad de zona | 60 |
| Balance de capacidad | 40 |
| **Total** | **100** |

#### Compatibilidad de zona

Se otorgan 60 puntos cuando la zona normalizada del registro coincide con la
zona normalizada del usuario.

Cuando no existe coincidencia, se otorgan 0 puntos.

La zona no constituye una restricción dura, ya que existen registros con valores
como `ANT` y `Bogotá` para los cuales no existe un usuario con coincidencia
exacta.

#### Balance de capacidad

El componente de capacidad busca favorecer a los usuarios con menor utilización
relativa de su capacidad.

La utilización se calcula como:

`carga_actual / capacidad_maxima`

y el puntaje como:

`40 * (1 - utilización)`

De esta manera, un usuario con menor utilización obtiene un mayor puntaje.

### 15.3 Fallback geográfico

Cuando existen usuarios elegibles con coincidencia exacta de zona, estos son
priorizados mediante el puntaje correspondiente.

Cuando no existe coincidencia, el motor permite utilizar usuarios de otras zonas
y continúa evaluando el balance de capacidad.

Para registros sin zona no se realiza ninguna inferencia geográfica. En estos
casos el componente de zona recibe 0 puntos.

No se realizarán transformaciones semánticas no sustentadas por los datos. Por
ejemplo, `ANT` no se interpretará automáticamente como `Antioquia` y `Bogotá`
no se interpretará automáticamente como `Centro`.

### 15.4 Desempate

Si dos o más usuarios obtienen el mismo puntaje total, se utilizarán los
siguientes criterios, en este orden:

1. Mayor capacidad disponible.
2. Menor `id` de usuario.

El desempate será determinista para garantizar que una misma entrada produzca
el mismo resultado.

### 15.5 Actualización de carga

Después de cada asignación, la `carga_actual` del usuario seleccionado se
incrementa en una unidad.

Por esta razón, el puntaje de capacidad se recalcula para cada registro y el
balance de carga evoluciona durante la ejecución.

### 15.6 Trazabilidad

Cada asignación simulada deberá conservar como mínimo:

- Registro asignado.
- Usuario seleccionado.
- Método utilizado.
- Puntaje total.
- Desglose del puntaje.
- Carga antes de la asignación.
- Carga después de la asignación.
- Candidatos considerados.
- Indicación de uso de fallback.
- Explicación de la decisión.

La simulación se utilizará para validar la metodología antes de implementarla
en el backend.

In [55]:
# Copia de los registros nuevos que serán procesados por el motor
registros_simulacion = registros_nuevos.copy()

# Copia de los usuarios elegibles
usuarios_simulacion = usuarios_operativos.copy()

# Aseguramos que las zonas utilizadas por el motor
# correspondan a las versiones normalizadas.
registros_simulacion["zona_normalizada"] = normalizar_texto(
    registros_simulacion["zona"]
)

usuarios_simulacion["zona_normalizada"] = normalizar_texto(
    usuarios_simulacion["zona"]
)

# La carga inicial representa el estado operativo
# al comenzar esta primera simulación.
usuarios_simulacion["carga_actual"] = 0

usuarios_simulacion["capacidad_disponible"] = (
    usuarios_simulacion["capacidad_maxima"]
    - usuarios_simulacion["carga_actual"]
)

display(
    usuarios_simulacion[
        [
            "id",
            "nombre",
            "capacidad_maxima",
            "carga_actual",
            "capacidad_disponible",
            "zona_normalizada",
            "segmento_experto",
        ]
    ].sort_values("id")
)

,id,nombre,capacidad_maxima,carga_actual,capacidad_disponible,zona_normalizada,segmento_experto
1,2,Carlos Mauricio Pardo,40.0,0,40.0,centro,corporativo
2,3,Juliana Ríos Gaviria,35.0,0,35.0,centro,corporativo
4,5,Paula Andrea Cárdenas,45.0,0,45.0,centro,pyme
7,8,Santiago Mejía Botero,45.0,0,45.0,centro,pyme
8,9,Mónica Álvarez Uribe,40.0,0,40.0,antioquia,industrial
9,10,Jorge Iván Arango,40.0,0,40.0,antioquia,industrial
11,12,Ricardo Lozano Pineda,38.0,0,38.0,occidente,corporativo
12,13,Catalina Vargas Nieto,38.0,0,38.0,occidente,pyme
13,14,Óscar Hernán Salazar,38.0,0,38.0,occidente,industrial
14,15,Marcela Fonseca Prieto,42.0,0,42.0,costa,pyme


In [56]:
def calcular_score_zona(zona_registro, zona_usuario):
    """
    Calcula el puntaje correspondiente a la compatibilidad geográfica.

    60 puntos:
        Cuando ambas zonas existen y coinciden.

    0 puntos:
        Cuando no existe coincidencia o alguna de las zonas está ausente.
    """
    
    if pd.isna(zona_registro) or pd.isna(zona_usuario):
        return 0.0

    if zona_registro == zona_usuario:
        return 60.0

    return 0.0


def calcular_score_carga(carga_actual, capacidad_maxima):
    """
    Calcula el puntaje asociado al balance de capacidad.
    """
    
    utilizacion = carga_actual / capacidad_maxima
    
    return 40.0 * (1 - utilizacion)


def calcular_score_total(
    zona_registro,
    zona_usuario,
    carga_actual,
    capacidad_maxima
):
    """
    Calcula el puntaje total de un usuario candidato.
    """
    
    score_zona = calcular_score_zona(
        zona_registro,
        zona_usuario
    )

    score_carga = calcular_score_carga(
        carga_actual,
        capacidad_maxima
    )

    return {
        "score_zona": score_zona,
        "score_carga": score_carga,
        "score_total": score_zona + score_carga,
    }

In [57]:
candidato_prueba = usuarios_simulacion.iloc[0]

resultado_score = calcular_score_total(
    zona_registro="centro",
    zona_usuario=candidato_prueba["zona_normalizada"],
    carga_actual=candidato_prueba["carga_actual"],
    capacidad_maxima=candidato_prueba["capacidad_maxima"],
)

print("Usuario:", candidato_prueba["nombre"])
print("Zona:", candidato_prueba["zona_normalizada"])
print("Score zona:", resultado_score["score_zona"])
print("Score carga:", resultado_score["score_carga"])
print("Score total:", resultado_score["score_total"])

Usuario: Carlos Mauricio Pardo
Zona: centro
Score zona: 60.0
Score carga: 40.0
Score total: 100.0


### 15.7 Evaluación de candidatos

Para cada registro, el motor evaluará todos los usuarios elegibles que aún
tengan capacidad disponible.

Cada candidato recibirá:

- Puntaje de compatibilidad de zona.
- Puntaje de balance de capacidad.
- Puntaje total.
- Capacidad disponible.
- Utilización actual.

Los candidatos serán ordenados de acuerdo con las reglas de selección y se
conservará el detalle necesario para explicar posteriormente la decisión.

In [58]:
def evaluar_candidatos(registro, usuarios_estado):
    """
    Evalúa todos los usuarios elegibles disponibles para un registro.

    No modifica el estado de los usuarios.
    Solamente calcula los puntajes y devuelve los candidatos ordenados.
    """

    candidatos = usuarios_estado[
        usuarios_estado["capacidad_disponible"] > 0
    ].copy()

    resultados = []

    for _, usuario in candidatos.iterrows():

        scores = calcular_score_total(
            zona_registro=registro["zona_normalizada"],
            zona_usuario=usuario["zona_normalizada"],
            carga_actual=usuario["carga_actual"],
            capacidad_maxima=usuario["capacidad_maxima"],
        )

        utilizacion = (
            usuario["carga_actual"]
            / usuario["capacidad_maxima"]
        )

        resultados.append(
            {
                "usuario_id": usuario["id"],
                "nombre": usuario["nombre"],
                "zona": usuario["zona_normalizada"],
                "capacidad_maxima": usuario["capacidad_maxima"],
                "carga_actual": usuario["carga_actual"],
                "capacidad_disponible": usuario["capacidad_disponible"],
                "utilizacion": utilizacion,
                "score_zona": scores["score_zona"],
                "score_carga": scores["score_carga"],
                "score_total": scores["score_total"],
            }
        )

    candidatos_evaluados = pd.DataFrame(resultados)

    if candidatos_evaluados.empty:
        return candidatos_evaluados

    # Orden de selección:
    # 1. Mayor score total
    # 2. Mayor capacidad disponible
    # 3. Menor ID de usuario
    candidatos_evaluados = candidatos_evaluados.sort_values(
        by=[
            "score_total",
            "capacidad_disponible",
            "usuario_id",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    ).reset_index(drop=True)

    return candidatos_evaluados

In [59]:
registro_prueba = registros_simulacion[
    registros_simulacion["zona_normalizada"] == "centro"
].iloc[0]

print("ID registro:", registro_prueba["id"])
print("Empresa:", registro_prueba["razon_social"])
print("Zona:", registro_prueba["zona"])

ID registro: 11
Empresa: Inversiones Santa Bárbara S.A.
Zona: Centro


In [60]:
candidatos_prueba = evaluar_candidatos(
    registro_prueba,
    usuarios_simulacion
)

display(candidatos_prueba)

,usuario_id,nombre,zona,capacidad_maxima,carga_actual,capacidad_disponible,utilizacion,score_zona,score_carga,score_total
0,5,Paula Andrea Cárdenas,centro,45.0,0,45.0,0.0,60.0,40.0,100.0
1,8,Santiago Mejía Botero,centro,45.0,0,45.0,0.0,60.0,40.0,100.0
2,2,Carlos Mauricio Pardo,centro,40.0,0,40.0,0.0,60.0,40.0,100.0
3,3,Juliana Ríos Gaviria,centro,35.0,0,35.0,0.0,60.0,40.0,100.0
4,15,Marcela Fonseca Prieto,costa,42.0,0,42.0,0.0,0.0,40.0,40.0
5,16,Julián Camargo Peña,costa,42.0,0,42.0,0.0,0.0,40.0,40.0
6,17,Daniela Quintero Ríos,costa,42.0,0,42.0,0.0,0.0,40.0,40.0
7,9,Mónica Álvarez Uribe,antioquia,40.0,0,40.0,0.0,0.0,40.0,40.0
8,10,Jorge Iván Arango,antioquia,40.0,0,40.0,0.0,0.0,40.0,40.0
9,12,Ricardo Lozano Pineda,occidente,38.0,0,38.0,0.0,0.0,40.0,40.0


In [61]:
registro_sin_coincidencia = registros_simulacion[
    registros_simulacion["zona_normalizada"] == "ant"
].iloc[0]

print("ID registro:", registro_sin_coincidencia["id"])
print("Empresa:", registro_sin_coincidencia["razon_social"])
print("Zona:", registro_sin_coincidencia["zona"])

ID registro: 9
Empresa: Soluciones Yumbo S.A.S.
Zona: ANT


In [62]:
candidatos_sin_coincidencia = evaluar_candidatos(
    registro_sin_coincidencia,
    usuarios_simulacion
)

display(candidatos_sin_coincidencia)

,usuario_id,nombre,zona,capacidad_maxima,carga_actual,capacidad_disponible,utilizacion,score_zona,score_carga,score_total
0,5,Paula Andrea Cárdenas,centro,45.0,0,45.0,0.0,0.0,40.0,40.0
1,8,Santiago Mejía Botero,centro,45.0,0,45.0,0.0,0.0,40.0,40.0
2,15,Marcela Fonseca Prieto,costa,42.0,0,42.0,0.0,0.0,40.0,40.0
3,16,Julián Camargo Peña,costa,42.0,0,42.0,0.0,0.0,40.0,40.0
4,17,Daniela Quintero Ríos,costa,42.0,0,42.0,0.0,0.0,40.0,40.0
5,2,Carlos Mauricio Pardo,centro,40.0,0,40.0,0.0,0.0,40.0,40.0
6,9,Mónica Álvarez Uribe,antioquia,40.0,0,40.0,0.0,0.0,40.0,40.0
7,10,Jorge Iván Arango,antioquia,40.0,0,40.0,0.0,0.0,40.0,40.0
8,12,Ricardo Lozano Pineda,occidente,38.0,0,38.0,0.0,0.0,40.0,40.0
9,13,Catalina Vargas Nieto,occidente,38.0,0,38.0,0.0,0.0,40.0,40.0


### 15.8 Ejecución de una asignación individual

La función de asignación selecciona el candidato con mayor puntuación, aplica los criterios de desempate y actualiza el estado operativo del usuario seleccionado.

La evaluación y la ejecución se mantienen separadas: `evaluar_candidatos()` calcula y ordena los candidatos, mientras que `asignar_registro()` materializa la decisión en la simulación.

Cada resultado conserva el puntaje, el desglose de criterios, la carga anterior y posterior, y la explicación de la decisión.


In [73]:
def asignar_registro(registro, usuarios_estado):
    """
    Asigna un registro a un usuario elegible disponible.

    Modifica la carga y la capacidad disponible del usuario seleccionado.
    Devuelve la asignación y su información de trazabilidad.
    """

    candidatos = evaluar_candidatos(
        registro,
        usuarios_estado,
    )

    if candidatos.empty:
        return {
            "asignado": False,
            "registro_id": int(registro["id"]),
            "razon": "No existen usuarios con capacidad disponible.",
        }

    # El primer candidato es el seleccionado según:
    # 1. Mayor score total.
    # 2. Mayor capacidad disponible.
    # 3. Menor ID de usuario.
    seleccionado = candidatos.iloc[0]

    usuario_id = int(seleccionado["usuario_id"])

    indice_usuario = usuarios_estado.index[
        usuarios_estado["id"] == usuario_id
    ][0]

    carga_antes = int(
        usuarios_estado.at[indice_usuario, "carga_actual"]
    )

    capacidad_antes = float(
        usuarios_estado.at[indice_usuario, "capacidad_disponible"]
    )

    # Actualizar el estado operativo.
    carga_despues = carga_antes + 1
    capacidad_despues = capacidad_antes - 1

    usuarios_estado.at[
        indice_usuario, "carga_actual"
    ] = carga_despues

    usuarios_estado.at[
        indice_usuario, "capacidad_disponible"
    ] = capacidad_despues

    # ---------------------------------------------------------
    # Explicación geográfica de la asignación
    # ---------------------------------------------------------

    zona_registro = registro["zona_normalizada"]
    zona_usuario = seleccionado["zona"]

    # Determinar si el usuario seleccionado tiene
    # coincidencia exacta de zona.
    coincidencia_zona = (
        pd.notna(zona_registro)
        and pd.notna(zona_usuario)
        and zona_registro == zona_usuario
    )

    # Comprobar si existe al menos un candidato elegible
    # cuya zona coincida exactamente con la zona del registro.
    if pd.isna(zona_registro):
        hay_candidato_en_zona = False
    else:
        hay_candidato_en_zona = (
            candidatos["zona"]
            .eq(zona_registro)
            .fillna(False)
            .any()
        )

    if pd.isna(zona_registro):

        estado_geografico = "zona_no_informada"

        explicacion_zona = (
            "El registro no tiene zona; no se aplicó "
            "puntuación geográfica."
        )

    elif coincidencia_zona:

        estado_geografico = "coincidencia_exacta"

        explicacion_zona = (
            "El usuario seleccionado tiene coincidencia "
            "exacta de zona."
        )

    elif not hay_candidato_en_zona:

        estado_geografico = "fallback_sin_candidato_en_zona"

        explicacion_zona = (
            "No había candidatos elegibles con coincidencia "
            "exacta de zona; se aplicó fallback geográfico."
        )

    else:

        estado_geografico = "seleccion_fuera_de_zona"

        explicacion_zona = (
            "Había candidatos elegibles de la misma zona, "
            "pero otro usuario obtuvo mayor puntuación total."
        )

    # ---------------------------------------------------------
    # Resultado de la asignación
    # ---------------------------------------------------------

    return {
        "asignado": True,
        "registro_id": int(registro["id"]),
        "razon_social": registro["razon_social"],
        "zona_registro_original": registro["zona"],
        "usuario_id": usuario_id,
        "usuario_nombre": seleccionado["nombre"],
        "metodo": "weighted_rules",
        "score_zona": float(seleccionado["score_zona"]),
        "score_carga": float(seleccionado["score_carga"]),
        "score_total": float(seleccionado["score_total"]),
        "carga_antes": carga_antes,
        "carga_despues": carga_despues,
        "capacidad_antes": capacidad_antes,
        "capacidad_despues": capacidad_despues,
        "utilizacion_antes": float(seleccionado["utilizacion"]),
        "coincidencia_zona": bool(coincidencia_zona),
        "fallback_geografico": (
            pd.notna(zona_registro)
            and not bool(coincidencia_zona)
        ),
        "estado_geografico": estado_geografico,
        "explicacion_zona": explicacion_zona,
        "razon": (
            f"Usuario seleccionado por mayor puntuación total "
            f"({seleccionado['score_total']:.2f}). "
            f"{explicacion_zona}"
        ),
    }

In [74]:
usuarios_prueba_individual = usuarios_simulacion.copy()

registro_prueba = registros_simulacion[
    registros_simulacion["zona_normalizada"] == "centro"
].iloc[0]

resultado_asignacion = asignar_registro(
    registro_prueba,
    usuarios_prueba_individual,
)

display(pd.DataFrame([resultado_asignacion]))

print("\n=== ESTADO DEL USUARIO DESPUÉS DE ASIGNAR ===")

display(
    usuarios_prueba_individual.loc[
        usuarios_prueba_individual["id"]
        == resultado_asignacion["usuario_id"],
        [
            "id",
            "nombre",
            "capacidad_maxima",
            "carga_actual",
            "capacidad_disponible",
        ],
    ]
)

,asignado,registro_id,razon_social,zona_registro_original,usuario_id,usuario_nombre,metodo,score_zona,score_carga,score_total,carga_antes,carga_despues,capacidad_antes,capacidad_despues,utilizacion_antes,coincidencia_zona,fallback_geografico,estado_geografico,explicacion_zona,razon
0,True,11,Inversiones Santa Bárbara S.A.,Centro,5,Paula Andrea Cárdenas,weighted_rules,60.0,40.0,100.0,0,1,45.0,44.0,0.0,True,False,coincidencia_exacta,El usuario seleccionado tiene coincidencia exa...,Usuario seleccionado por mayor puntuación tota...



=== ESTADO DEL USUARIO DESPUÉS DE ASIGNAR ===


,id,nombre,capacidad_maxima,carga_actual,capacidad_disponible
4,5,Paula Andrea Cárdenas,45.0,1,44.0


### 15.9 Simulación secuencial de asignaciones

Se procesarán los registros nuevos en un orden determinista, utilizando su identificador como criterio de orden.

Después de cada asignación, se actualizará la carga del usuario seleccionado. Por tanto, la evaluación del siguiente registro utilizará el estado operativo resultante de las decisiones anteriores.

La simulación generará dos resultados:

* Un historial de asignaciones con puntajes y explicaciones.
* Un estado final de capacidad por usuario.

Esta simulación representa una ejecución inicial con carga previa igual a cero para los usuarios elegibles. No debe interpretarse como una medición de la carga comercial real existente antes de la ejecución.


In [75]:
# Reiniciar el estado operativo para una ejecución limpia.
usuarios_estado_simulacion = usuarios_operativos.copy()

usuarios_estado_simulacion["carga_actual"] = 0

usuarios_estado_simulacion["capacidad_disponible"] = (
    usuarios_estado_simulacion["capacidad_maxima"]
    - usuarios_estado_simulacion["carga_actual"]
)

# Orden reproducible de procesamiento.
registros_ordenados = (
    registros_simulacion
    .sort_values("id")
    .copy()
)

historial_asignaciones = []

for _, registro in registros_ordenados.iterrows():

    resultado = asignar_registro(
        registro,
        usuarios_estado_simulacion,
    )

    historial_asignaciones.append(resultado)

# Convertir el historial en una tabla.
asignaciones_simuladas = pd.DataFrame(
    historial_asignaciones
)

print("Registros procesados:", len(registros_ordenados))
print(
    "Asignaciones exitosas:",
    asignaciones_simuladas["asignado"].sum()
)
print(
    "Registros sin asignación:",
    (~asignaciones_simuladas["asignado"]).sum()
)

display(
    asignaciones_simuladas[
        [
            "registro_id",
            "razon_social",
            "usuario_id",
            "usuario_nombre",
            "score_zona",
            "score_carga",
            "score_total",
            "carga_antes",
            "carga_despues",
            "fallback_geografico",
            "razon",
        ]
    ].head(15)
)

Registros procesados: 71
Asignaciones exitosas: 71
Registros sin asignación: 0


,registro_id,razon_social,usuario_id,usuario_nombre,score_zona,score_carga,score_total,carga_antes,carga_despues,fallback_geografico,razon
0,1,Servicios Girardot S.A.,15,Marcela Fonseca Prieto,60.0,40.000000,100.000000,0,1,False,Usuario seleccionado por mayor puntuación tota...
1,2,COMERCIALIZADORA ANDINA SAS,12,Ricardo Lozano Pineda,60.0,40.000000,100.000000,0,1,False,Usuario seleccionado por mayor puntuación tota...
2,3,Industrias Andina S.A.S.,5,Paula Andrea Cárdenas,0.0,40.000000,40.000000,0,1,False,Usuario seleccionado por mayor puntuación tota...
3,4,Constructora Caribe Azul Ltda.,13,Catalina Vargas Nieto,60.0,40.000000,100.000000,0,1,False,Usuario seleccionado por mayor puntuación tota...
4,5,Distribuidora La Esperanza Ltda.,9,Mónica Álvarez Uribe,60.0,40.000000,100.000000,0,1,False,Usuario seleccionado por mayor puntuación tota...
5,6,Suministros Vallecaucana Ltda.,16,Julián Camargo Peña,60.0,40.000000,100.000000,0,1,False,Usuario seleccionado por mayor puntuación tota...
6,7,Grupo El Poblado S.A.,14,Óscar Hernán Salazar,60.0,40.000000,100.000000,0,1,False,Usuario seleccionado por mayor puntuación tota...
7,8,Manufacturas El Dorado S.A.,10,Jorge Iván Arango,60.0,40.000000,100.000000,0,1,False,Usuario seleccionado por mayor puntuación tota...
8,9,Soluciones Yumbo S.A.S.,8,Santiago Mejía Botero,0.0,40.000000,40.000000,0,1,True,Usuario seleccionado por mayor puntuación tota...
9,10,Textiles La Esperanza S.A.,9,Mónica Álvarez Uribe,60.0,39.000000,99.000000,1,2,False,Usuario seleccionado por mayor puntuación tota...


In [86]:
# =========================================================
# RESUMEN Y VALIDACIÓN FINAL DE CARGA
# =========================================================

print("=== RESUMEN FINAL DE CARGA ===")

resumen_carga = (
    usuarios_estado_simulacion[
        [
            "id",
            "nombre",
            "zona_normalizada",
            "capacidad_maxima",
            "carga_actual",
            "capacidad_disponible",
        ]
    ]
    .sort_values(
        ["carga_actual", "id"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(resumen_carga)

# ---------------------------------------------------------
# Totales
# ---------------------------------------------------------

total_asignaciones = int(
    usuarios_estado_simulacion["carga_actual"].sum()
)

capacidad_inicial = float(
    usuarios_estado_simulacion["capacidad_maxima"].sum()
)

capacidad_disponible_total = float(
    usuarios_estado_simulacion["capacidad_disponible"].sum()
)

capacidad_consumida = (
    capacidad_inicial
    - capacidad_disponible_total
)

print(
    "Total de asignaciones:",
    total_asignaciones,
)

print(
    "Capacidad inicial total:",
    capacidad_inicial,
)

print(
    "Capacidad disponible total:",
    capacidad_disponible_total,
)

print(
    "Capacidad consumida:",
    capacidad_consumida,
)

# ---------------------------------------------------------
# Estadísticas de carga
# ---------------------------------------------------------

print("\n=== ESTADÍSTICAS DE CARGA ===")

print(
    "Carga mínima:",
    usuarios_estado_simulacion["carga_actual"].min(),
)

print(
    "Carga máxima:",
    usuarios_estado_simulacion["carga_actual"].max(),
)

print(
    "Carga promedio:",
    usuarios_estado_simulacion["carga_actual"].mean(),
)

print(
    "Desviación estándar:",
    usuarios_estado_simulacion["carga_actual"].std(),
)

# ---------------------------------------------------------
# Validaciones de consistencia
# ---------------------------------------------------------

print("\n=== VALIDACIONES ===")

# La carga acumulada debe coincidir con las asignaciones exitosas.
assert (
    total_asignaciones
    == int(asignaciones_simuladas["asignado"].sum())
)

# La capacidad restante debe ser:
# capacidad inicial - carga asignada.
assert (
    capacidad_disponible_total
    == capacidad_inicial - total_asignaciones
)

# Ningún usuario puede tener capacidad disponible negativa.
assert (
    usuarios_estado_simulacion["capacidad_disponible"] >= 0
).all()

# La carga nunca puede superar la capacidad máxima.
assert (
    usuarios_estado_simulacion["carga_actual"]
    <= usuarios_estado_simulacion["capacidad_maxima"]
).all()

print("✓ La carga coincide con las asignaciones realizadas.")
print("✓ La capacidad disponible es consistente.")
print("✓ Ningún usuario excede su capacidad.")
print("✓ La simulación mantiene un estado operativo válido.")

print("\n=== RESULTADO ===")

print(
    f"Se procesaron {len(registros_ordenados)} registros."
)

print(
    f"Se realizaron {total_asignaciones} asignaciones."
)

print(
    f"Quedaron {capacidad_disponible_total:.0f} unidades "
    "de capacidad disponibles."
)

=== RESUMEN FINAL DE CARGA ===


,id,nombre,zona_normalizada,capacidad_maxima,carga_actual,capacidad_disponible
0,12,Ricardo Lozano Pineda,occidente,38.0,8,30.0
1,13,Catalina Vargas Nieto,occidente,38.0,7,31.0
2,14,Óscar Hernán Salazar,occidente,38.0,7,31.0
3,5,Paula Andrea Cárdenas,centro,45.0,6,39.0
4,2,Carlos Mauricio Pardo,centro,40.0,5,35.0
5,8,Santiago Mejía Botero,centro,45.0,5,40.0
6,9,Mónica Álvarez Uribe,antioquia,40.0,5,35.0
7,10,Jorge Iván Arango,antioquia,40.0,5,35.0
8,15,Marcela Fonseca Prieto,costa,42.0,5,37.0
9,16,Julián Camargo Peña,costa,42.0,5,37.0


Total de asignaciones: 71
Capacidad inicial total: 520.0
Capacidad disponible total: 449.0
Capacidad consumida: 71.0

=== ESTADÍSTICAS DE CARGA ===
Carga mínima: 4
Carga máxima: 8
Carga promedio: 5.461538461538462
Desviación estándar: 1.1982893790305562

=== VALIDACIONES ===
✓ La carga coincide con las asignaciones realizadas.
✓ La capacidad disponible es consistente.
✓ Ningún usuario excede su capacidad.
✓ La simulación mantiene un estado operativo válido.

=== RESULTADO ===
Se procesaron 71 registros.
Se realizaron 71 asignaciones.
Quedaron 449 unidades de capacidad disponibles.


In [87]:
# Cada registro debe aparecer una sola vez en la simulación.
assert asignaciones_simuladas["registro_id"].is_unique

# Todas las asignaciones deben tener usuario seleccionado.
asignaciones_exitosas = asignaciones_simuladas[
    asignaciones_simuladas["asignado"]
].copy()

assert asignaciones_exitosas["usuario_id"].notna().all()

# Ningún usuario puede superar su capacidad máxima.
assert (
    usuarios_estado_simulacion["carga_actual"]
    <= usuarios_estado_simulacion["capacidad_maxima"]
).all()

# La carga debe coincidir con las asignaciones recibidas.
carga_por_asignaciones = (
    asignaciones_exitosas
    .groupby("usuario_id")
    .size()
)

for _, usuario in usuarios_estado_simulacion.iterrows():
    carga_observada = int(
        carga_por_asignaciones.get(usuario["id"], 0)
    )

    assert carga_observada == int(usuario["carga_actual"])

print("Todas las validaciones de la simulación fueron satisfactorias.")

Todas las validaciones de la simulación fueron satisfactorias.


In [88]:
print("=== DISTRIBUCIÓN POR USUARIO ===")

display(
    asignaciones_simuladas
    .groupby(["usuario_id", "usuario_nombre"])
    .agg(
        registros_asignados=("registro_id", "count"),
        score_promedio=("score_total", "mean"),
        capacidad_antes_promedio=("capacidad_antes", "mean"),
    )
    .sort_values("registros_asignados", ascending=False)
    .reset_index()
)

=== DISTRIBUCIÓN POR USUARIO ===


,usuario_id,usuario_nombre,registros_asignados,score_promedio,capacidad_antes_promedio
0,12,Ricardo Lozano Pineda,8,96.315789,34.5
1,13,Catalina Vargas Nieto,7,96.842105,35.0
2,14,Óscar Hernán Salazar,7,96.842105,35.0
3,5,Paula Andrea Cárdenas,6,77.777778,42.5
4,2,Carlos Mauricio Pardo,5,74.000000,38.0
5,15,Marcela Fonseca Prieto,5,86.095238,40.0
6,9,Mónica Álvarez Uribe,5,86.000000,38.0
7,8,Santiago Mejía Botero,5,86.222222,43.0
8,10,Jorge Iván Arango,5,74.000000,38.0
9,17,Daniela Quintero Ríos,5,86.095238,40.0


In [89]:
print("=== USO DE FALLBACK GEOGRÁFICO ===")

print(
    asignaciones_simuladas["fallback_geografico"]
    .value_counts(dropna=False)
)

print("\n=== ASIGNACIONES SIN COINCIDENCIA GEOGRÁFICA ===")

display(
    asignaciones_simuladas.loc[
        asignaciones_simuladas["fallback_geografico"],
        [
            "registro_id",
            "razon_social",
            "zona_registro_original",
            "usuario_id",
            "usuario_nombre",
            "score_zona",
            "score_carga",
            "score_total",
            "explicacion_zona",
        ],
    ]
)

=== USO DE FALLBACK GEOGRÁFICO ===
fallback_geografico
False    60
True     11
Name: count, dtype: int64

=== ASIGNACIONES SIN COINCIDENCIA GEOGRÁFICA ===


,registro_id,razon_social,zona_registro_original,usuario_id,usuario_nombre,score_zona,score_carga,score_total,explicacion_zona
8,9,Soluciones Yumbo S.A.S.,ANT,8,Santiago Mejía Botero,0.0,40.000000,40.000000,No había candidatos elegibles con coincidencia...
13,14,Textiles La Candelaria S. en C.,ANT,18,Felipe Andrade Mora,0.0,40.000000,40.000000,No había candidatos elegibles con coincidencia...
37,38,Industrias Tequendama S.A.S.,ANT,18,Felipe Andrade Mora,0.0,38.857143,38.857143,No había candidatos elegibles con coincidencia...
39,40,Soluciones Magdalena Ltda.,Bogotá,2,Carlos Mauricio Pardo,0.0,38.000000,38.000000,No había candidatos elegibles con coincidencia...
41,42,Soluciones Chicamocha S. en C.,Bogotá,18,Felipe Andrade Mora,0.0,37.714286,37.714286,No había candidatos elegibles con coincidencia...
42,43,Comercializadora La Esperanza S.A.S.,ANT,5,Paula Andrea Cárdenas,0.0,37.333333,37.333333,No había candidatos elegibles con coincidencia...
47,48,Comercializadora Magdalena S.A.S.,ANT,15,Marcela Fonseca Prieto,0.0,37.142857,37.142857,No había candidatos elegibles con coincidencia...
52,53,Grupo Guadalupe S.A.S.,Bogotá,2,Carlos Mauricio Pardo,0.0,37.000000,37.000000,No había candidatos elegibles con coincidencia...
54,55,Manufacturas Pacífico Sur S.A.,Bogotá,3,Juliana Ríos Gaviria,0.0,36.571429,36.571429,No había candidatos elegibles con coincidencia...
65,66,Agropecuaria Caribe Azul Ltda.,ANT,9,Mónica Álvarez Uribe,0.0,36.000000,36.000000,No había candidatos elegibles con coincidencia...


In [90]:
cargas = usuarios_estado_simulacion["carga_actual"]

print("Usuarios elegibles:", len(usuarios_estado_simulacion))
print("Carga mínima:", cargas.min())
print("Carga máxima:", cargas.max())
print("Carga promedio:", cargas.mean())
print("Desviación estándar:", cargas.std())

Usuarios elegibles: 13
Carga mínima: 4
Carga máxima: 8
Carga promedio: 5.461538461538462
Desviación estándar: 1.1982893790305562


In [91]:
asignaciones_simuladas[
    asignaciones_simuladas["estado_geografico"] == "seleccion_fuera_de_zona"
][
    [
        "registro_id",
        "razon_social",
        "zona_registro_original",
        "usuario_id",
        "usuario_nombre",
        "score_zona",
        "score_carga",
        "score_total",
        "explicacion_zona",
    ]
]

,registro_id,razon_social,zona_registro_original,usuario_id,usuario_nombre,score_zona,score_carga,score_total,explicacion_zona


In [92]:
asignaciones_simuladas["estado_geografico"].value_counts()

estado_geografico
coincidencia_exacta               55
fallback_sin_candidato_en_zona    11
zona_no_informada                  5
Name: count, dtype: int64

In [93]:
print("=== VALIDACIÓN DE LA SIMULACIÓN ===")

total_registros = len(registros_simulacion)

total_asignados = asignaciones_simuladas["asignado"].sum()

total_no_asignados = (
    total_registros - total_asignados
)

print(f"Registros evaluados: {total_registros}")
print(f"Registros asignados: {total_asignados}")
print(f"Registros no asignados: {total_no_asignados}")

print("\n=== CAPACIDAD ===")

print(
    f"Capacidad inicial total: "
    f"{usuarios_operativos['capacidad_maxima'].sum():.0f}"
)

print(
    f"Registros asignados: "
    f"{total_asignados}"
)

print(
    f"Capacidad restante total: "
    f"{usuarios_operativos['capacidad_disponible'].sum():.0f}"
)

=== VALIDACIÓN DE LA SIMULACIÓN ===
Registros evaluados: 71
Registros asignados: 71
Registros no asignados: 0

=== CAPACIDAD ===
Capacidad inicial total: 520
Registros asignados: 71
Capacidad restante total: 520


In [94]:
print("=== USUARIOS QUE EXCEDIERON CAPACIDAD ===")

usuarios_excedidos = usuarios_operativos[
    usuarios_operativos["capacidad_disponible"] < 0
][
    [
        "id",
        "nombre",
        "capacidad_maxima",
        "carga_actual",
        "capacidad_disponible",
    ]
]

display(usuarios_excedidos)

print(
    f"Usuarios excedidos: {len(usuarios_excedidos)}"
)

=== USUARIOS QUE EXCEDIERON CAPACIDAD ===


,id,nombre,capacidad_maxima,carga_actual,capacidad_disponible


Usuarios excedidos: 0


In [95]:
print("=== DISTRIBUCIÓN DE CARGA ===")

distribucion_carga = (
    usuarios_operativos[
        [
            "id",
            "nombre",
            "capacidad_maxima",
            "carga_actual",
            "capacidad_disponible",
        ]
    ]
    .sort_values("carga_actual", ascending=False)
    .reset_index(drop=True)
)

display(distribucion_carga)

=== DISTRIBUCIÓN DE CARGA ===


,id,nombre,capacidad_maxima,carga_actual,capacidad_disponible
0,2,Carlos Mauricio Pardo,40.0,0,40.0
1,3,Juliana Ríos Gaviria,35.0,0,35.0
2,5,Paula Andrea Cárdenas,45.0,0,45.0
3,8,Santiago Mejía Botero,45.0,0,45.0
4,9,Mónica Álvarez Uribe,40.0,0,40.0
5,10,Jorge Iván Arango,40.0,0,40.0
6,12,Ricardo Lozano Pineda,38.0,0,38.0
7,13,Catalina Vargas Nieto,38.0,0,38.0
8,14,Óscar Hernán Salazar,38.0,0,38.0
9,15,Marcela Fonseca Prieto,42.0,0,42.0


In [96]:
print("=== ESTADO ACTUAL DE LA SIMULACIÓN ===")

print(
    "Asignaciones registradas:",
    int(asignaciones_simuladas["asignado"].sum())
)

print(
    "Carga acumulada:",
    int(usuarios_estado_simulacion["carga_actual"].sum())
)

print(
    "Capacidad disponible:",
    usuarios_estado_simulacion["capacidad_disponible"].sum()
)

print("\n=== CARGA POR USUARIO ===")

display(
    usuarios_estado_simulacion[
        [
            "id",
            "nombre",
            "capacidad_maxima",
            "carga_actual",
            "capacidad_disponible",
        ]
    ]
    .sort_values(
        ["carga_actual", "id"],
        ascending=[False, True],
    )
)

=== ESTADO ACTUAL DE LA SIMULACIÓN ===
Asignaciones registradas: 71
Carga acumulada: 71
Capacidad disponible: 449.0

=== CARGA POR USUARIO ===


,id,nombre,capacidad_maxima,carga_actual,capacidad_disponible
11,12,Ricardo Lozano Pineda,38.0,8,30.0
12,13,Catalina Vargas Nieto,38.0,7,31.0
13,14,Óscar Hernán Salazar,38.0,7,31.0
4,5,Paula Andrea Cárdenas,45.0,6,39.0
1,2,Carlos Mauricio Pardo,40.0,5,35.0
7,8,Santiago Mejía Botero,45.0,5,40.0
8,9,Mónica Álvarez Uribe,40.0,5,35.0
9,10,Jorge Iván Arango,40.0,5,35.0
14,15,Marcela Fonseca Prieto,42.0,5,37.0
15,16,Julián Camargo Peña,42.0,5,37.0


In [97]:
print("=== VALIDACIÓN DEL ESTADO OPERATIVO ===")

print(
    "Carga acumulada:",
    int(
        usuarios_estado_simulacion["carga_actual"].sum()
    )
)

print(
    "Capacidad disponible:",
    usuarios_estado_simulacion[
        "capacidad_disponible"
    ].sum()
)

print(
    "Usuarios excedidos:",
    int(
        (
            usuarios_estado_simulacion["carga_actual"]
            > usuarios_estado_simulacion["capacidad_maxima"]
        ).sum()
    )
)

=== VALIDACIÓN DEL ESTADO OPERATIVO ===
Carga acumulada: 71
Capacidad disponible: 449.0
Usuarios excedidos: 0


In [98]:
# =========================================================
# VALIDACIÓN MATEMÁTICA DE SCORES
# =========================================================

print("=== VALIDACIÓN DE SCORES ===")

# 1. El score total debe ser la suma de los componentes.
validacion_score_total = np.isclose(
    asignaciones_simuladas["score_total"],
    asignaciones_simuladas["score_zona"]
    + asignaciones_simuladas["score_carga"],
)

# 2. Score geográfico dentro del rango definido.
validacion_score_zona = (
    asignaciones_simuladas["score_zona"].between(0, 60)
)

# 3. Score de carga dentro del rango definido.
validacion_score_carga = (
    asignaciones_simuladas["score_carga"].between(0, 40)
)

# 4. Score total dentro del rango definido.
validacion_score_total_rango = (
    asignaciones_simuladas["score_total"].between(0, 100)
)

print(
    "Score total = score zona + score carga:",
    int(validacion_score_total.sum()),
    "/",
    len(asignaciones_simuladas),
)

print(
    "Score zona dentro de [0, 60]:",
    int(validacion_score_zona.sum()),
    "/",
    len(asignaciones_simuladas),
)

print(
    "Score carga dentro de [0, 40]:",
    int(validacion_score_carga.sum()),
    "/",
    len(asignaciones_simuladas),
)

print(
    "Score total dentro de [0, 100]:",
    int(validacion_score_total_rango.sum()),
    "/",
    len(asignaciones_simuladas),
)

# Validación global.
assert validacion_score_total.all()
assert validacion_score_zona.all()
assert validacion_score_carga.all()
assert validacion_score_total_rango.all()

print("\n✓ Todos los scores son matemáticamente consistentes.")

=== VALIDACIÓN DE SCORES ===
Score total = score zona + score carga: 71 / 71
Score zona dentro de [0, 60]: 71 / 71
Score carga dentro de [0, 40]: 71 / 71
Score total dentro de [0, 100]: 71 / 71

✓ Todos los scores son matemáticamente consistentes.


In [99]:
# =========================================================
# VALIDACIÓN DE ELEGIBILIDAD
# =========================================================

print("=== VALIDACIÓN DE ELEGIBILIDAD ===")

usuarios_por_id = usuarios.set_index("id")

asignaciones_con_usuario = asignaciones_simuladas[
    asignaciones_simuladas["asignado"]
].copy()

asignaciones_con_usuario["usuario_activo"] = (
    asignaciones_con_usuario["usuario_id"]
    .map(usuarios_por_id["activo"])
)

asignaciones_con_usuario["usuario_rol"] = (
    asignaciones_con_usuario["usuario_id"]
    .map(usuarios_por_id["rol"])
)

asignaciones_con_usuario["capacidad_maxima"] = (
    asignaciones_con_usuario["usuario_id"]
    .map(usuarios_por_id["capacidad_maxima"])
)

roles_validos = {"vendedor", "lider"}

validacion_activo = (
    asignaciones_con_usuario["usuario_activo"] == True
)

validacion_rol = (
    asignaciones_con_usuario["usuario_rol"]
    .isin(roles_validos)
)

validacion_capacidad = (
    asignaciones_con_usuario["capacidad_maxima"].notna()
    &
    (asignaciones_con_usuario["capacidad_maxima"] > 0)
)

print(
    "Usuarios activos:",
    int(validacion_activo.sum()),
    "/",
    len(asignaciones_con_usuario),
)

print(
    "Roles válidos:",
    int(validacion_rol.sum()),
    "/",
    len(asignaciones_con_usuario),
)

print(
    "Capacidad máxima válida:",
    int(validacion_capacidad.sum()),
    "/",
    len(asignaciones_con_usuario),
)

assert validacion_activo.all()
assert validacion_rol.all()
assert validacion_capacidad.all()

print("\n✓ Todas las asignaciones respetan las reglas de elegibilidad.")

=== VALIDACIÓN DE ELEGIBILIDAD ===
Usuarios activos: 71 / 71
Roles válidos: 71 / 71
Capacidad máxima válida: 71 / 71

✓ Todas las asignaciones respetan las reglas de elegibilidad.


In [100]:
# =========================================================
# VALIDACIÓN DE CAPACIDAD POR USUARIO
# =========================================================

print("=== VALIDACIÓN DE CAPACIDAD POR USUARIO ===")

validacion_capacidad_final = (
    usuarios_estado_simulacion["carga_actual"]
    <= usuarios_estado_simulacion["capacidad_maxima"]
)

display(
    usuarios_estado_simulacion[
        [
            "id",
            "nombre",
            "capacidad_maxima",
            "carga_actual",
            "capacidad_disponible",
        ]
    ]
    .assign(
        capacidad_valida=validacion_capacidad_final
    )
)

assert validacion_capacidad_final.all()

print("\n✓ Ningún usuario supera su capacidad máxima.")

=== VALIDACIÓN DE CAPACIDAD POR USUARIO ===


,id,nombre,capacidad_maxima,carga_actual,capacidad_disponible,capacidad_valida
1,2,Carlos Mauricio Pardo,40.0,5,35.0,True
2,3,Juliana Ríos Gaviria,35.0,4,31.0,True
4,5,Paula Andrea Cárdenas,45.0,6,39.0,True
7,8,Santiago Mejía Botero,45.0,5,40.0,True
8,9,Mónica Álvarez Uribe,40.0,5,35.0,True
9,10,Jorge Iván Arango,40.0,5,35.0,True
11,12,Ricardo Lozano Pineda,38.0,8,30.0,True
12,13,Catalina Vargas Nieto,38.0,7,31.0,True
13,14,Óscar Hernán Salazar,38.0,7,31.0,True
14,15,Marcela Fonseca Prieto,42.0,5,37.0,True



✓ Ningún usuario supera su capacidad máxima.


In [101]:
# =========================================================
# INSPECCIÓN DE UNA ASIGNACIÓN
# =========================================================

registro_id_prueba = 1

registro_prueba = registros_simulacion[
    registros_simulacion["id"] == registro_id_prueba
].iloc[0]

print("=== REGISTRO SELECCIONADO ===")

display(
    registro_prueba[
        [
            "id",
            "razon_social",
            "zona",
            "zona_normalizada",
            "sector",
            "empleados",
            "ingresos_estimados",
            "estado",
        ]
    ].to_frame().T
)

# Crear una copia del estado operativo.
usuarios_estado_prueba = usuarios_operativos.copy()

usuarios_estado_prueba["carga_actual"] = 0

usuarios_estado_prueba["capacidad_disponible"] = (
    usuarios_estado_prueba["capacidad_maxima"]
    - usuarios_estado_prueba["carga_actual"]
)

# Evaluar candidatos sin modificar el estado.
candidatos_prueba = evaluar_candidatos(
    registro_prueba,
    usuarios_estado_prueba,
)

# Agregar posición según el orden definido por el algoritmo.
candidatos_prueba = (
    candidatos_prueba
    .reset_index(drop=True)
    .copy()
)

candidatos_prueba["posicion"] = (
    candidatos_prueba.index + 1
)

print("\n=== CANDIDATOS ORDENADOS ===")

display(
    candidatos_prueba[
        [
            "posicion",
            "usuario_id",
            "nombre",
            "zona",
            "capacidad_maxima",
            "carga_actual",
            "capacidad_disponible",
            "score_zona",
            "score_carga",
            "score_total",
        ]
    ]
)

print("\n=== CANDIDATO SELECCIONADO ===")

seleccionado_prueba = candidatos_prueba.iloc[0]

print(
    f"Usuario seleccionado: "
    f"{int(seleccionado_prueba['usuario_id'])} "
    f"- {seleccionado_prueba['nombre']}"
)

print(
    f"Score zona: "
    f"{seleccionado_prueba['score_zona']:.2f}"
)

print(
    f"Score carga: "
    f"{seleccionado_prueba['score_carga']:.2f}"
)

print(
    f"Score total: "
    f"{seleccionado_prueba['score_total']:.2f}"
)

=== REGISTRO SELECCIONADO ===


,id,razon_social,zona,zona_normalizada,sector,empleados,ingresos_estimados,estado
0,1,Servicios Girardot S.A.,Costa,costa,Metalmecánica,8.0,1866000000.0,nuevo



=== CANDIDATOS ORDENADOS ===


,posicion,usuario_id,nombre,zona,capacidad_maxima,carga_actual,capacidad_disponible,score_zona,score_carga,score_total
0,1,15,Marcela Fonseca Prieto,costa,42.0,0,42.0,60.0,40.0,100.0
1,2,16,Julián Camargo Peña,costa,42.0,0,42.0,60.0,40.0,100.0
2,3,17,Daniela Quintero Ríos,costa,42.0,0,42.0,60.0,40.0,100.0
3,4,5,Paula Andrea Cárdenas,centro,45.0,0,45.0,0.0,40.0,40.0
4,5,8,Santiago Mejía Botero,centro,45.0,0,45.0,0.0,40.0,40.0
5,6,2,Carlos Mauricio Pardo,centro,40.0,0,40.0,0.0,40.0,40.0
6,7,9,Mónica Álvarez Uribe,antioquia,40.0,0,40.0,0.0,40.0,40.0
7,8,10,Jorge Iván Arango,antioquia,40.0,0,40.0,0.0,40.0,40.0
8,9,12,Ricardo Lozano Pineda,occidente,38.0,0,38.0,0.0,40.0,40.0
9,10,13,Catalina Vargas Nieto,occidente,38.0,0,38.0,0.0,40.0,40.0



=== CANDIDATO SELECCIONADO ===
Usuario seleccionado: 15 - Marcela Fonseca Prieto
Score zona: 60.00
Score carga: 40.00
Score total: 100.00


In [102]:
# =========================================================
# RECONSTRUCCIÓN DE UNA ASIGNACIÓN REAL
# =========================================================

registro_id_prueba = 48

resultado_real = asignaciones_simuladas[
    asignaciones_simuladas["registro_id"] == registro_id_prueba
].iloc[0]

print("=== ASIGNACIÓN REGISTRADA ===")

print(
    f"Registro: {int(resultado_real['registro_id'])}"
)

print(
    f"Empresa: {resultado_real['razon_social']}"
)

print(
    f"Zona del registro: "
    f"{resultado_real['zona_registro_original']}"
)

print(
    f"Usuario seleccionado: "
    f"{int(resultado_real['usuario_id'])} "
    f"- {resultado_real['usuario_nombre']}"
)

print(
    f"Score zona: "
    f"{resultado_real['score_zona']:.2f}"
)

print(
    f"Score carga: "
    f"{resultado_real['score_carga']:.2f}"
)

print(
    f"Score total: "
    f"{resultado_real['score_total']:.2f}"
)

print(
    f"Carga antes: "
    f"{resultado_real['carga_antes']}"
)

print(
    f"Carga después: "
    f"{resultado_real['carga_despues']}"
)

print(
    f"Estado geográfico: "
    f"{resultado_real['estado_geografico']}"
)

print(
    f"Explicación: "
    f"{resultado_real['explicacion_zona']}"
)

=== ASIGNACIÓN REGISTRADA ===
Registro: 48
Empresa: Comercializadora Magdalena S.A.S.
Zona del registro: ANT
Usuario seleccionado: 15 - Marcela Fonseca Prieto
Score zona: 0.00
Score carga: 37.14
Score total: 37.14
Carga antes: 3
Carga después: 4
Estado geográfico: fallback_sin_candidato_en_zona
Explicación: No había candidatos elegibles con coincidencia exacta de zona; se aplicó fallback geográfico.


In [103]:
# =========================================================
# VALIDACIÓN GLOBAL DE LA SELECCIÓN
# =========================================================

print("=== VALIDACIÓN GLOBAL DE SELECCIÓN ===")

# Estado operativo independiente para reconstruir
# exactamente la misma ejecución.
usuarios_estado_validacion = usuarios_operativos.copy()

usuarios_estado_validacion["carga_actual"] = 0

usuarios_estado_validacion["capacidad_disponible"] = (
    usuarios_estado_validacion["capacidad_maxima"]
    - usuarios_estado_validacion["carga_actual"]
)

errores_seleccion = []

for _, registro in registros_ordenados.iterrows():

    candidatos = evaluar_candidatos(
        registro,
        usuarios_estado_validacion,
    )

    resultado_real = asignaciones_simuladas[
        asignaciones_simuladas["registro_id"]
        == registro["id"]
    ].iloc[0]

    # Si no hay candidatos, validar que el motor
    # tampoco haya asignado el registro.
    if candidatos.empty:

        if bool(resultado_real["asignado"]):
            errores_seleccion.append({
                "registro_id": int(registro["id"]),
                "tipo_error": "Asignación sin candidatos",
            })

        continue

    # El primer candidato debe ser el seleccionado
    # por el motor.
    candidato_esperado = candidatos.iloc[0]

    usuario_esperado = int(
        candidato_esperado["usuario_id"]
    )

    usuario_real = int(
        resultado_real["usuario_id"]
    )

    if usuario_esperado != usuario_real:

        errores_seleccion.append({
            "registro_id": int(registro["id"]),
            "usuario_esperado": usuario_esperado,
            "usuario_real": usuario_real,
            "score_esperado": float(
                candidato_esperado["score_total"]
            ),
            "score_real": float(
                resultado_real["score_total"]
            ),
        })

    # Reproducir exactamente la modificación de estado
    # realizada por asignar_registro().
    indice_usuario = usuarios_estado_validacion.index[
        usuarios_estado_validacion["id"]
        == usuario_real
    ][0]

    usuarios_estado_validacion.at[
        indice_usuario,
        "carga_actual"
    ] += 1

    usuarios_estado_validacion.at[
        indice_usuario,
        "capacidad_disponible"
    ] -= 1


# ---------------------------------------------------------
# Resultado
# ---------------------------------------------------------

print(
    "Decisiones verificadas:",
    len(registros_ordenados),
)

print(
    "Decisiones correctas:",
    len(registros_ordenados)
    - len(errores_seleccion),
)

print(
    "Errores encontrados:",
    len(errores_seleccion),
)

assert len(errores_seleccion) == 0

print(
    "\n✓ Las 71 decisiones fueron reproducidas "
    "correctamente."
)

print(
    "✓ El usuario seleccionado coincide con "
    "el primer candidato del ranking."
)

=== VALIDACIÓN GLOBAL DE SELECCIÓN ===
Decisiones verificadas: 71
Decisiones correctas: 71
Errores encontrados: 0

✓ Las 71 decisiones fueron reproducidas correctamente.
✓ El usuario seleccionado coincide con el primer candidato del ranking.


In [104]:
# =========================================================
# PRUEBA DE DESEMPATE 1
# Mismo score_total → mayor capacidad disponible
# =========================================================

print("=== PRUEBA DE DESEMPATE 1 ===")

# Crear dos candidatos artificiales.
candidatos_empate = pd.DataFrame([
    {
        "usuario_id": 10,
        "nombre": "Usuario A",
        "capacidad_maxima": 40,
        "carga_actual": 10,
        "capacidad_disponible": 30,
        "score_zona": 60,
        "score_carga": 30,
        "score_total": 90,
    },
    {
        "usuario_id": 20,
        "nombre": "Usuario B",
        "capacidad_maxima": 50,
        "carga_actual": 20,
        "capacidad_disponible": 30,
        "score_zona": 60,
        "score_carga": 30,
        "score_total": 90,
    },
])

# Orden definido por Weighted Rules:
# 1. mayor score_total
# 2. mayor capacidad disponible
# 3. menor ID
ranking_empate = (
    candidatos_empate
    .sort_values(
        [
            "score_total",
            "capacidad_disponible",
            "usuario_id",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(
    ranking_empate[
        [
            "usuario_id",
            "nombre",
            "capacidad_disponible",
            "score_total",
        ]
    ]
)

seleccionado = ranking_empate.iloc[0]

print(
    f"Usuario seleccionado: "
    f"{int(seleccionado['usuario_id'])} "
    f"- {seleccionado['nombre']}"
)

assert int(seleccionado["usuario_id"]) == 10

print(
    "\n✓ El primer criterio empató correctamente."
)

print(
    "✓ El desempate se resolvió mediante "
    "capacidad disponible."
)

=== PRUEBA DE DESEMPATE 1 ===


,usuario_id,nombre,capacidad_disponible,score_total
0,10,Usuario A,30,90
1,20,Usuario B,30,90


Usuario seleccionado: 10 - Usuario A

✓ El primer criterio empató correctamente.
✓ El desempate se resolvió mediante capacidad disponible.


In [105]:
# =========================================================
# PRUEBA A
# Mismo score_total → mayor capacidad disponible
# =========================================================

candidatos_empate = pd.DataFrame([
    {
        "usuario_id": 10,
        "nombre": "Usuario A",
        "capacidad_disponible": 20,
        "score_total": 90,
    },
    {
        "usuario_id": 20,
        "nombre": "Usuario B",
        "capacidad_disponible": 30,
        "score_total": 90,
    },
])

ranking_empate = (
    candidatos_empate
    .sort_values(
        [
            "score_total",
            "capacidad_disponible",
            "usuario_id",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(ranking_empate)

seleccionado = ranking_empate.iloc[0]

print(
    "Usuario seleccionado:",
    int(seleccionado["usuario_id"]),
    "-",
    seleccionado["nombre"],
)

assert int(seleccionado["usuario_id"]) == 20

print(
    "\n✓ Mismo score_total correctamente resuelto."
)

print(
    "✓ Ganó el candidato con mayor capacidad disponible."
)

,usuario_id,nombre,capacidad_disponible,score_total
0,20,Usuario B,30,90
1,10,Usuario A,20,90


Usuario seleccionado: 20 - Usuario B

✓ Mismo score_total correctamente resuelto.
✓ Ganó el candidato con mayor capacidad disponible.


In [106]:
# =========================================================
# PRUEBA B
# Mismo score_total + misma capacidad → menor ID
# =========================================================

candidatos_empate_id = pd.DataFrame([
    {
        "usuario_id": 10,
        "nombre": "Usuario A",
        "capacidad_disponible": 30,
        "score_total": 90,
    },
    {
        "usuario_id": 20,
        "nombre": "Usuario B",
        "capacidad_disponible": 30,
        "score_total": 90,
    },
])

ranking_empate_id = (
    candidatos_empate_id
    .sort_values(
        [
            "score_total",
            "capacidad_disponible",
            "usuario_id",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(ranking_empate_id)

seleccionado = ranking_empate_id.iloc[0]

print(
    "Usuario seleccionado:",
    int(seleccionado["usuario_id"]),
    "-",
    seleccionado["nombre"],
)

assert int(seleccionado["usuario_id"]) == 10

print(
    "\n✓ Score y capacidad empatados correctamente."
)

print(
    "✓ Ganó el candidato con menor ID."
)

,usuario_id,nombre,capacidad_disponible,score_total
0,10,Usuario A,30,90
1,20,Usuario B,30,90


Usuario seleccionado: 10 - Usuario A

✓ Score y capacidad empatados correctamente.
✓ Ganó el candidato con menor ID.


## 16. Conclusiones y contrato del algoritmo Weighted Rules

La fase de exploración y validación permitió establecer las condiciones de calidad de los datos y definir una metodología determinista para la asignación de nuevos registros comerciales.

Sobre los 167 registros disponibles, se identificaron **71 registros en estado `nuevo`** como candidatos para asignación. Estos registros no presentan actividad histórica asociada, por lo que constituyen el conjunto utilizado para validar el motor de asignación.

Para la evaluación se estableció como fecha de referencia el **3 de octubre de 2026**. A partir de esta fecha se determinaron los usuarios disponibles considerando estado activo, rol asignable, ausencia vigente y capacidad disponible.

El conjunto resultante contiene **13 usuarios elegibles**, con una capacidad total inicial de **520 asignaciones**. Después de procesar los 71 registros, se consumieron 71 unidades de capacidad y permanecieron **449 unidades disponibles**. Ningún usuario superó su capacidad máxima.

### Validación del algoritmo

La metodología `Weighted Rules` fue validada mediante diferentes niveles de comprobación:

* **71 de 71** asignaciones reprodujeron correctamente la lógica de selección.
* Todos los `score_total` corresponden a la suma de `score_zona` y `score_carga`.
* `score_zona` permanece dentro del rango `[0, 60]`.
* `score_carga` permanece dentro del rango `[0, 40]`.
* `score_total` permanece dentro del rango `[0, 100]`.
* Todas las asignaciones cumplen las reglas de elegibilidad.
* La carga acumulada coincide con las asignaciones realizadas.
* La capacidad disponible se mantiene consistente después de cada asignación.
* No se presentaron asignaciones por encima de la capacidad máxima.

### Validación geográfica

La zona se utiliza como un criterio ponderado y no como una restricción absoluta.

Las coincidencias exactas de zona reciben **60 puntos**, mientras que el balance de carga aporta hasta **40 puntos**.

En la simulación de los 71 registros:

| Resultado geográfico                           | Registros |
| ---------------------------------------------- | --------: |
| Coincidencia exacta                            |        55 |
| Fallback por ausencia de candidato en zona     |        11 |
| Zona no informada                              |         5 |
| Selección fuera de zona con candidatos en zona |         0 |

Los resultados muestran que el motor mantiene la preferencia geográfica definida sin convertir la zona en una restricción rígida.

Cuando un registro contiene una zona para la cual no existe un candidato elegible, se utiliza un **fallback geográfico**, permitiendo que el registro sea asignado utilizando principalmente el criterio de balance de carga.

No se realizan conversiones semánticas no definidas por el negocio. Por ejemplo, valores como `ANT` y `Antioquia` no se consideran equivalentes únicamente por interpretación del algoritmo.

### Contrato funcional de Weighted Rules

El comportamiento esperado del algoritmo queda definido de la siguiente manera:

**1. Determinar elegibilidad**

Un usuario puede recibir un registro únicamente si:

* Está activo.
* Tiene rol `vendedor` o `lider`.
* No presenta una ausencia vigente en la fecha de evaluación.
* Tiene una capacidad máxima definida.
* Tiene capacidad disponible mayor que cero.

Estas condiciones son restricciones duras y no generan puntuación.

**2. Calcular puntuación**

Para cada candidato elegible:

```text
score_zona =
    60 si existe coincidencia exacta de zona
     0 en cualquier otro caso

score_carga =
    40 × (1 - carga_actual / capacidad_maxima)

score_total =
    score_zona + score_carga
```

**3. Ordenar candidatos**

La selección debe ser determinista utilizando:

```text
score_total DESC
capacidad_disponible DESC
usuario_id ASC
```

Es decir:

1. Se selecciona el mayor `score_total`.
2. Si existe empate, gana quien tenga mayor capacidad disponible.
3. Si persiste el empate, gana el usuario con menor ID.

**4. Actualizar el estado**

Después de una asignación:

```text
carga_actual = carga_actual + 1
capacidad_disponible = capacidad_disponible - 1
```

El estado actualizado debe utilizarse para la siguiente asignación.

**5. Registrar trazabilidad**

Cada asignación debe conservar información suficiente para responder:

> ¿Por qué este registro terminó asignado a este usuario?

Como mínimo, la trazabilidad debe incluir:

* Registro asignado.
* Usuario seleccionado.
* Método utilizado.
* Score de zona.
* Score de carga.
* Score total.
* Carga antes y después.
* Capacidad antes y después.
* Estado geográfico.
* Explicación de la decisión.
* Información relevante de los candidatos evaluados.

### Resultado de la fase

La exploración de datos permitió identificar inconsistencias y valores faltantes sin realizar correcciones silenciosas sobre la información original. Las transformaciones aplicadas para comparación fueron limitadas a normalización de texto, principalmente eliminación de espacios y estandarización de mayúsculas/minúsculas.

La metodología `Weighted Rules` queda definida como un algoritmo **determinista, reproducible y auditable**, con reglas explícitas de elegibilidad, puntuación, desempate, actualización de capacidad y trazabilidad.

La simulación sobre los 71 registros nuevos demuestra que el algoritmo puede procesar el conjunto completo manteniendo un estado operativo consistente y respetando las restricciones definidas.

Por tanto, esta fase establece el **contrato funcional del motor de asignación** que será utilizado como referencia para la implementación posterior en Java/Spring Boot.

La implementación deberá conservar estas reglas y resultados conceptuales, evitando introducir cambios de comportamiento durante la migración del prototipo en Python al backend.
